In [ ]:
# 依赖包安装（若已配置，可跳过）
!pip install -U tqsdk pandas numpy statsmodels arch matplotlib openpyxl


In [ ]:
# 导入依赖库与画图基本配置
import os
import getpass
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqsdk import TqApi, TqAuth

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from arch import arch_model

warnings.filterwarnings("ignore")

# Jupyter 显示设置
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

# 输出目录
OUTPUT_DIR = Path("./sr_traditional_volatility_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("输出目录：", OUTPUT_DIR.resolve())


In [ ]:
# 郑商所白糖主力合约（KQ.m@CZCE.SR）设定及账户配置
SYMBOL = "KQ.m@CZCE.SR"
DURATION_SECONDS = 24 * 60 * 60   # 日线
DATA_LENGTH = 8000

# 优先从环境变量载入凭证
TQ_USER = os.getenv("TQ_USER")
TQ_PASSWORD = os.getenv("TQ_PASSWORD")

if not TQ_USER:
    TQ_USER = input("请输入天勤账号：")

if not TQ_PASSWORD:
    TQ_PASSWORD = getpass.getpass("请输入天勤密码：")

print("研究标的：", SYMBOL)
print("周期：日线")
print("样本长度请求：", DATA_LENGTH)


In [ ]:
# 通过天勤API同步获取白糖主力日线历史行情数据
api = TqApi(auth=TqAuth(TQ_USER, TQ_PASSWORD))

klines = api.get_kline_serial(
    SYMBOL,
    duration_seconds=DURATION_SECONDS,
    data_length=DATA_LENGTH
)

# 等待API数据同步
# 轮询推进API行情数据
while not api.is_serial_ready(klines):
    api.wait_update()

raw = klines.copy()

api.close()

print("原始数据行数：", len(raw))
raw.head()


In [ ]:
# 原始日线数据清洗与字段对齐
df = raw.copy()

df["datetime"] = pd.to_datetime(df["datetime"], unit="ns")

df = df.sort_values("datetime").reset_index(drop=True)

keep_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "open_oi",
    "close_oi"
]

df = df[keep_cols].copy()

df = df.dropna(subset=["close"])
df = df[df["close"] > 0].reset_index(drop=True)

df["date"] = df["datetime"].dt.date

print("清洗后数据行数：", len(df))
print("起始日期：", df["datetime"].min())
print("结束日期：", df["datetime"].max())

df.head()


In [ ]:
# 计算对数收益率与百分比收益率
df["log_price"] = np.log(df["close"])
df["ret"] = df["log_price"].diff()
df["ret_pct"] = df["ret"] * 100

df = df.dropna(subset=["ret", "ret_pct"]).reset_index(drop=True)

# 换月时跳空点检测与标记
df["abs_ret_pct"] = df["ret_pct"].abs()
extreme_cut = df["abs_ret_pct"].quantile(0.999)
df["is_extreme_ret"] = df["abs_ret_pct"] > extreme_cut

print("收益率样本行数：", len(df))
print("极端收益阈值 99.9% 分位：", extreme_cut)

df[["datetime", "close", "ret", "ret_pct", "is_extreme_ret"]].head()


In [ ]:
# 保存清洗后的日线数据至CSV
price_file = OUTPUT_DIR / "sr_main_daily_cleaned.csv"
df.to_csv(price_file, index=False, encoding="utf-8-sig")

print("已保存：", price_file)


In [ ]:
# 绘制价格、收益率及平方收益率时序图（初步判断平稳性与聚集特征）
plt.figure(figsize=(14, 5))
plt.plot(df["datetime"], df["close"])
plt.title("CZCE Sugar Main Continuous Close Price")
plt.xlabel("Date")
plt.ylabel("Close")
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(df["datetime"], df["ret_pct"])
plt.title("Daily Log Return of CZCE Sugar Main Continuous Contract (%)")
plt.xlabel("Date")
plt.ylabel("Return (%)")
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(df["datetime"], df["ret_pct"] ** 2)
plt.title("Squared Daily Return of CZCE Sugar Main Continuous Contract")
plt.xlabel("Date")
plt.ylabel("Squared return")
plt.grid(True)
plt.show()


In [ ]:
# 收益率分布特征描述统计
ret = df["ret_pct"].dropna()

desc = pd.DataFrame({
    "metric": [
        "count",
        "mean",
        "std",
        "min",
        "1%",
        "5%",
        "25%",
        "50%",
        "75%",
        "95%",
        "99%",
        "max",
        "skew",
        "kurtosis"
    ],
    "value": [
        ret.count(),
        ret.mean(),
        ret.std(),
        ret.min(),
        ret.quantile(0.01),
        ret.quantile(0.05),
        ret.quantile(0.25),
        ret.quantile(0.50),
        ret.quantile(0.75),
        ret.quantile(0.95),
        ret.quantile(0.99),
        ret.max(),
        ret.skew(),
        ret.kurtosis()
    ]
})

desc_file = OUTPUT_DIR / "01_return_descriptive_stats.csv"
desc.to_csv(desc_file, index=False, encoding="utf-8-sig")

desc


In [ ]:
# ADF单位根平稳性检验
def run_adf_test(series, name):
    series = pd.Series(series).dropna()
    result = adfuller(series, autolag="AIC")

    out = {
        "series": name,
        "adf_stat": result[0],
        "p_value": result[1],
        "used_lag": result[2],
        "n_obs": result[3],
        "critical_1%": result[4].get("1%"),
        "critical_5%": result[4].get("5%"),
        "critical_10%": result[4].get("10%"),
        "decision_5pct": "stationary" if result[1] < 0.05 else "non_stationary_or_inconclusive"
    }
    return out

adf_results = pd.DataFrame([
    run_adf_test(df["log_price"], "log_price"),
    run_adf_test(df["ret_pct"], "ret_pct")
])

adf_file = OUTPUT_DIR / "02_adf_tests.csv"
adf_results.to_csv(adf_file, index=False, encoding="utf-8-sig")

adf_results


In [ ]:
# Ljung-Box自相关检验（评估收益率及其平方项）
def run_ljungbox(series, name, lags=(5, 10, 20, 30)):
    series = pd.Series(series).dropna()
    lb = acorr_ljungbox(series, lags=list(lags), return_df=True)
    lb = lb.reset_index().rename(columns={"index": "lag"})
    lb.insert(0, "series", name)
    lb["decision_5pct"] = np.where(
        lb["lb_pvalue"] < 0.05,
        "reject_no_autocorr",
        "fail_to_reject_no_autocorr"
    )
    return lb

lb_ret = run_ljungbox(df["ret_pct"], "ret_pct")
lb_ret_sq = run_ljungbox(df["ret_pct"] ** 2, "ret_pct_squared")

lb_raw = pd.concat([lb_ret, lb_ret_sq], ignore_index=True)

lb_file = OUTPUT_DIR / "03_ljungbox_raw_returns.csv"
lb_raw.to_csv(lb_file, index=False, encoding="utf-8-sig")

lb_raw


In [ ]:
# ARCH-LM条件异方差效应检验
def run_arch_lm(series, name, nlags=10):
    series = pd.Series(series).dropna()
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(series, nlags=nlags)
    return {
        "series": name,
        "nlags": nlags,
        "lm_stat": lm_stat,
        "lm_pvalue": lm_pvalue,
        "f_stat": f_stat,
        "f_pvalue": f_pvalue,
        "decision_5pct": "arch_effect_exists" if lm_pvalue < 0.05 else "no_arch_effect_detected"
    }

arch_lm_raw = pd.DataFrame([
    run_arch_lm(df["ret_pct"], "ret_pct", nlags=5),
    run_arch_lm(df["ret_pct"], "ret_pct", nlags=10),
    run_arch_lm(df["ret_pct"], "ret_pct", nlags=20)
])

arch_lm_file = OUTPUT_DIR / "04_arch_lm_raw_returns.csv"
arch_lm_raw.to_csv(arch_lm_file, index=False, encoding="utf-8-sig")

arch_lm_raw


In [ ]:
# 传统波动率建模可行性综合判断
adf_ret_p = adf_results.loc[adf_results["series"] == "ret_pct", "p_value"].iloc[0]
lb_sq_p10 = lb_raw[(lb_raw["series"] == "ret_pct_squared") & (lb_raw["lag"] == 10)]["lb_pvalue"].iloc[0]
arch_p10 = arch_lm_raw[arch_lm_raw["nlags"] == 10]["lm_pvalue"].iloc[0]

print("========== 传统建模前判断 ==========")
print(f"ADF(ret_pct) p-value        = {adf_ret_p:.6f}")
print(f"Ljung-Box(ret_pct^2, lag10) = {lb_sq_p10:.6f}")
print(f"ARCH-LM(ret_pct, lag10)     = {arch_p10:.6f}")

if adf_ret_p < 0.05:
    print("1. 收益率通过平稳性检验：可以对收益率建模。")
else:
    print("1. 收益率未通过平稳性检验：需要谨慎检查数据。")

if lb_sq_p10 < 0.05:
    print("2. 平方收益率存在显著自相关：存在波动率聚集。")
else:
    print("2. 平方收益率未发现显著自相关：波动率聚集不强。")

if arch_p10 < 0.05:
    print("3. ARCH-LM 显著：存在条件异方差，适合 GARCH 类模型。")
else:
    print("3. ARCH-LM 不显著：GARCH 必要性下降。")


In [ ]:
# 设置待拟合收益率变量（百分比收益率）
y = df["ret_pct"].dropna()

def fit_arch_model(model_name, y, mean, vol, p=1, o=0, q=1, dist="normal", lags=None):
    """
    拟合 arch 包中的单变量波动率模型。

    参数含义：
    model_name：模型名称，仅用于输出记录
    y         ：收益率序列，建议使用百分比收益率
    mean      ：均值模型，例如 Constant, AR
    vol       ：波动率模型，例如 ARCH, GARCH, EGARCH
    p, o, q   ：波动率模型阶数
    dist      ：残差分布，例如 normal, t
    lags      ：均值方程 AR 滞后阶数；mean='AR' 时使用
    """
    try:
        if mean == "AR":
            am = arch_model(
                y,
                mean=mean,
                lags=lags,
                vol=vol,
                p=p,
                o=o,
                q=q,
                dist=dist,
                rescale=False
            )
        else:
            am = arch_model(
                y,
                mean=mean,
                vol=vol,
                p=p,
                o=o,
                q=q,
                dist=dist,
                rescale=False
            )

        res = am.fit(disp="off")
        print(f"{model_name}: fitted successfully")
        return res

    except Exception as e:
        print(f"{model_name}: failed -> {e}")
        return None


In [ ]:
# 拟合传统单变量GARCH族波动率模型组
models = {}

# 1. ARCH(5)-Student-t
models["ARCH5_t"] = fit_arch_model(
    model_name="ARCH5_t",
    y=y,
    mean="Constant",
    vol="ARCH",
    p=5,
    o=0,
    q=0,
    dist="t"
)

models["GARCH11_Normal"] = fit_arch_model(
    model_name="GARCH11_Normal",
    y=y,
    mean="Constant",
    vol="GARCH",
    p=1,
    o=0,
    q=1,
    dist="normal"
)

models["GARCH11_t"] = fit_arch_model(
    model_name="GARCH11_t",
    y=y,
    mean="Constant",
    vol="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t"
)

models["EGARCH11_t"] = fit_arch_model(
    model_name="EGARCH11_t",
    y=y,
    mean="Constant",
    vol="EGARCH",
    p=1,
    o=1,
    q=1,
    dist="t"
)

models["GJR_GARCH11_t"] = fit_arch_model(
    model_name="GJR_GARCH11_t",
    y=y,
    mean="Constant",
    vol="GARCH",
    p=1,
    o=1,
    q=1,
    dist="t"
)

# 运行Ljung-Box自相关检验
models["AR1_GARCH11_t"] = fit_arch_model(
    model_name="AR1_GARCH11_t",
    y=y,
    mean="AR",
    lags=1,
    vol="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t"
)

# 删除拟合失败的模型
models = {k: v for k, v in models.items() if v is not None}

print("成功拟合模型：")
list(models.keys())


In [ ]:
# 保存模型估计Summary结果至本地文件
summary_dir = OUTPUT_DIR / "model_summaries"
summary_dir.mkdir(exist_ok=True)

for name, res in models.items():
    summary_text = str(res.summary())
    file_path = summary_dir / f"{name}_summary.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(summary_text)

print("模型摘要已保存到：", summary_dir.resolve())


In [ ]:
# 整理与对比模型估计系数、信息准则（AIC/BIC）及波动持续性
def get_param(res, key):
    try:
        return res.params.get(key, np.nan)
    except Exception:
        return np.nan

param_rows = []

for name, res in models.items():
    params = res.params

    row = {
        "model": name,
        "loglikelihood": res.loglikelihood,
        "aic": res.aic,
        "bic": res.bic,
        "nobs": res.nobs,
        "mu": get_param(res, "mu"),
        "omega": get_param(res, "omega"),
        "alpha[1]": get_param(res, "alpha[1]"),
        "gamma[1]": get_param(res, "gamma[1]"),
        "beta[1]": get_param(res, "beta[1]"),
        "nu": get_param(res, "nu")
    }

    if not np.isnan(row["alpha[1]"]) and not np.isnan(row["beta[1]"]):
        row["alpha_plus_beta"] = row["alpha[1]"] + row["beta[1]"]
    else:
        row["alpha_plus_beta"] = np.nan

    if not np.isnan(row["alpha[1]"]) and not np.isnan(row["beta[1]"]) and not np.isnan(row["gamma[1]"]):
        row["gjr_persistence_approx"] = row["alpha[1]"] + row["beta[1]"] + 0.5 * row["gamma[1]"]
    else:
        row["gjr_persistence_approx"] = np.nan

    param_rows.append(row)

model_params = pd.DataFrame(param_rows).sort_values("aic").reset_index(drop=True)

params_file = OUTPUT_DIR / "05_model_parameters.csv"
model_params.to_csv(params_file, index=False, encoding="utf-8-sig")

model_params


In [ ]:
# 定义标准化残差自相关与ARCH效应诊断函数
def diagnose_standardized_residuals(res, model_name, lags=(5, 10, 20, 30)):
    std_resid = pd.Series(res.std_resid).replace([np.inf, -np.inf], np.nan).dropna()
    std_resid_sq = std_resid ** 2

    lb_z = acorr_ljungbox(std_resid, lags=list(lags), return_df=True)
    lb_z2 = acorr_ljungbox(std_resid_sq, lags=list(lags), return_df=True)

    arch_lm_10 = het_arch(std_resid, nlags=10)

    row = {
        "model": model_name,
        "n_std_resid": len(std_resid),
        "std_resid_mean": std_resid.mean(),
        "std_resid_std": std_resid.std(),
        "std_resid_skew": std_resid.skew(),
        "std_resid_kurtosis": std_resid.kurtosis(),
        "ARCH_LM_lag10_pvalue": arch_lm_10[1]
    }

    for lag in lags:
        row[f"LB_z_lag{lag}_pvalue"] = lb_z.loc[lag, "lb_pvalue"]
        row[f"LB_z2_lag{lag}_pvalue"] = lb_z2.loc[lag, "lb_pvalue"]

    lb_z_10 = row["LB_z_lag10_pvalue"]
    lb_z2_10 = row["LB_z2_lag10_pvalue"]
    arch_p = row["ARCH_LM_lag10_pvalue"]

    row["mean_equation_pass_lag10"] = lb_z_10 > 0.05
    row["vol_equation_pass_lag10"] = lb_z2_10 > 0.05
    row["arch_lm_pass_lag10"] = arch_p > 0.05

    if row["mean_equation_pass_lag10"] and row["vol_equation_pass_lag10"] and row["arch_lm_pass_lag10"]:
        row["traditional_decision"] = "PASS"
    else:
        row["traditional_decision"] = "FAIL_OR_NEEDS_IMPROVEMENT"

    return row


In [ ]:
# 对所有拟合模型运行标准化残差自检验
diag_rows = []

for name, res in models.items():
    diag_rows.append(diagnose_standardized_residuals(res, name))

diagnostics = pd.DataFrame(diag_rows)

model_compare = diagnostics.merge(
    model_params,
    on="model",
    how="left"
)

model_compare = model_compare.sort_values(
    by=["traditional_decision", "aic"],
    ascending=[True, True]
).reset_index(drop=True)

diag_file = OUTPUT_DIR / "06_model_diagnostics.csv"
model_compare.to_csv(diag_file, index=False, encoding="utf-8-sig")

model_compare


In [ ]:
# 生成模型拟合系数与残差诊断综合对比结果表
core_cols = [
    "model",
    "aic",
    "bic",
    "alpha[1]",
    "gamma[1]",
    "beta[1]",
    "nu",
    "alpha_plus_beta",
    "gjr_persistence_approx",
    "LB_z_lag10_pvalue",
    "LB_z2_lag10_pvalue",
    "ARCH_LM_lag10_pvalue",
    "traditional_decision"
]

core_result = model_compare[core_cols].copy()

core_file = OUTPUT_DIR / "07_core_model_comparison.csv"
core_result.to_csv(core_file, index=False, encoding="utf-8-sig")

core_result


In [ ]:
# 绘制各模型拟合的条件波动率时序对比图
vol_df = pd.DataFrame({"datetime": df.loc[y.index, "datetime"].values})

for name, res in models.items():
    vol = pd.Series(res.conditional_volatility, index=y.index)
    vol_df[name] = vol.values

vol_file = OUTPUT_DIR / "08_conditional_volatility.csv"
vol_df.to_csv(vol_file, index=False, encoding="utf-8-sig")

plt.figure(figsize=(14, 6))

for name in models.keys():
    plt.plot(vol_df["datetime"], vol_df[name], label=name, linewidth=1)

plt.title("Estimated Conditional Volatility of CZCE Sugar Main Contract")
plt.xlabel("Date")
plt.ylabel("Conditional volatility (%)")
plt.legend()
plt.grid(True)
plt.show()

vol_df.head()


In [ ]:
# 绘制残差诊断通过的最优波动率模型标准化残差图
pass_models = core_result[core_result["traditional_decision"] == "PASS"]

if len(pass_models) > 0:
    best_model_name = pass_models.sort_values("aic").iloc[0]["model"]
else:
    best_model_name = core_result.sort_values("aic").iloc[0]["model"]

best_res = models[best_model_name]
best_std_resid = pd.Series(best_res.std_resid, index=y.index).replace([np.inf, -np.inf], np.nan).dropna()

print("传统流程下选出的候选最佳模型：", best_model_name)

plt.figure(figsize=(14, 5))
plt.plot(df.loc[best_std_resid.index, "datetime"], best_std_resid)
plt.title(f"Standardized Residuals: {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Standardized residual")
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(df.loc[best_std_resid.index, "datetime"], best_std_resid ** 2)
plt.title(f"Squared Standardized Residuals: {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Squared standardized residual")
plt.grid(True)
plt.show()


In [ ]:
# 传统诊断指标下最优模型选择决策总结
print("========== 白糖主连传统波动率建模结论 ==========\n")

print("一、数据基础判断")
print(f"ADF(ret_pct) p-value = {adf_ret_p:.6f}")
print(f"Ljung-Box(ret_pct^2, lag10) p-value = {lb_sq_p10:.6f}")
print(f"ARCH-LM(ret_pct, lag10) p-value = {arch_p10:.6f}")

if adf_ret_p < 0.05 and lb_sq_p10 < 0.05 and arch_p10 < 0.05:
    print("结论：收益率平稳，平方收益率存在自相关，ARCH-LM 显著，具备 GARCH 类建模基础。\n")
else:
    print("结论：传统 GARCH 建模基础不完全充分，需要检查数据或样本区间。\n")

print("二、模型比较")
display(core_result)

print("\n三、传统诊断通过情况")
num_pass = (core_result["traditional_decision"] == "PASS").sum()
print(f"通过传统诊断的模型数量：{num_pass}")

if num_pass > 0:
    print(f"传统诊断下候选最佳模型：{best_model_name}")
    print("选择依据：该模型通过标准化残差 Ljung-Box、标准化残差平方 Ljung-Box、ARCH-LM 检验，并在通过模型中 AIC 较低。")
else:
    print(f"没有模型完全通过传统诊断。当前 AIC 最低模型为：{best_model_name}")
    print("含义：传统模型组仍可能遗漏波动率结构，后续应考虑更高阶模型、不同误差分布、样本处理或 Hong-Lee 类更一般检验。")

print("\n四、解释口径")
print("如果 LB_z_lag10_pvalue < 0.05：说明标准化残差本身仍有方向性自相关，均值方程可能不充分。")
print("如果 LB_z2_lag10_pvalue < 0.05：说明标准化残差平方仍有自相关，方差方程没有完全解释波动率聚集。")
print("如果 ARCH_LM_lag10_pvalue < 0.05：说明标准化残差仍有剩余 ARCH 效应。")
print("如果 Student-t 模型 AIC/BIC 明显优于 Normal：说明厚尾分布更适合白糖收益率。")
print("如果 GJR / EGARCH 的 gamma 显著且诊断改善：说明正负冲击对白糖未来波动率影响可能不对称。")


In [ ]:
# 提取标准化残差与条件方差，准备高阶特征核检验（Hong-Lee）
hong_lee_input = pd.DataFrame({
    "datetime": df.loc[best_std_resid.index, "datetime"].values,
    "std_resid": best_std_resid.values,
    "std_resid_sq": best_std_resid.values ** 2
})

hong_lee_input_file = OUTPUT_DIR / "09_hong_lee_input_standardized_residuals.csv"
hong_lee_input.to_csv(hong_lee_input_file, index=False, encoding="utf-8-sig")

print("Hong-Lee 后续检验输入文件已保存：", hong_lee_input_file.resolve())

hong_lee_input.head()


In [ ]:
# 锁定EGARCH(1,1)-t作为后续风险分析目标模型
TARGET_MODEL_NAME = "EGARCH11_t"

required_vars = ["df", "y", "models", "OUTPUT_DIR"]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} 不存在。请先运行前面传统建模流程对应 Cell。")

if TARGET_MODEL_NAME not in models:
    raise ValueError(
        f"{TARGET_MODEL_NAME} 不在 models 中。当前可用模型为：{list(models.keys())}"
    )

egarch_res = models[TARGET_MODEL_NAME]

print("目标模型：", TARGET_MODEL_NAME)
print("样本量：", egarch_res.nobs)
print("AIC：", egarch_res.aic)
print("BIC：", egarch_res.bic)

egarch_res.summary()


In [ ]:
# 提取并展示目标EGARCH模型的估计参数
params = egarch_res.params
pvalues = egarch_res.pvalues

egarch_param_table = pd.DataFrame({
    "parameter": params.index,
    "estimate": params.values,
    "p_value": [pvalues.get(k, np.nan) for k in params.index]
})

egarch_param_table["significant_5pct"] = egarch_param_table["p_value"] < 0.05

param_file = OUTPUT_DIR / "10_egarch11_t_parameters.csv"
egarch_param_table.to_csv(param_file, index=False, encoding="utf-8-sig")

egarch_param_table


In [ ]:
# 解析EGARCH模型参数的经济与统计学意义
def safe_get_param(name):
    return float(params.get(name, np.nan))

def safe_get_pvalue(name):
    return float(pvalues.get(name, np.nan))

mu = safe_get_param("mu")
omega = safe_get_param("omega")
alpha1 = safe_get_param("alpha[1]")
gamma1 = safe_get_param("gamma[1]")
beta1 = safe_get_param("beta[1]")
nu = safe_get_param("nu")

gamma_p = safe_get_pvalue("gamma[1]")
alpha_p = safe_get_pvalue("alpha[1]")
beta_p = safe_get_pvalue("beta[1]")
nu_p = safe_get_pvalue("nu")

print("========== EGARCH(1,1)-t 参数解释 ==========\n")

print(f"mu        = {mu:.6f}")
print(f"omega     = {omega:.6f}")
print(f"alpha[1]  = {alpha1:.6f}, p-value = {alpha_p:.6f}")
print(f"gamma[1]  = {gamma1:.6f}, p-value = {gamma_p:.6f}")
print(f"beta[1]   = {beta1:.6f}, p-value = {beta_p:.6f}")
print(f"nu        = {nu:.6f}, p-value = {nu_p:.6f}")

print("\n---------- 解释 ----------")

if not np.isnan(alpha1):
    if alpha_p < 0.05:
        print("1. alpha[1] 显著：冲击幅度会显著影响白糖未来波动率。")
    else:
        print("1. alpha[1] 不显著：冲击幅度对未来波动率的影响在统计上不强。")

if not np.isnan(beta1):
    if beta_p < 0.05:
        print("2. beta[1] 显著：白糖波动率具有显著持续性。")
    else:
        print("2. beta[1] 不显著：波动率持续性证据不强。")

    if abs(beta1) > 0.90:
        print("   beta[1] 接近 1：高波动状态可能持续较久。")
    elif abs(beta1) > 0.70:
        print("   beta[1] 较高：波动率有一定持续性。")
    else:
        print("   beta[1] 不高：波动率冲击衰减可能较快。")

if not np.isnan(gamma1):
    if gamma_p < 0.05:
        print("3. gamma[1] 显著：正负冲击对白糖未来波动率的影响存在非对称性。")
        if gamma1 < 0:
            print("   在 arch 包 EGARCH 设定下，gamma[1] < 0 通常表示负冲击更容易推高未来波动率。")
        elif gamma1 > 0:
            print("   在 arch 包 EGARCH 设定下，gamma[1] > 0 通常表示正冲击更容易推高未来波动率。")
    else:
        print("3. gamma[1] 不显著：非对称波动证据不强。")

if not np.isnan(nu):
    print("4. nu 是 Student-t 自由度。")
    if nu < 5:
        print("   nu 较低：白糖收益率厚尾非常明显，极端波动风险较高。")
    elif nu < 10:
        print("   nu 偏低：白糖收益率存在明显厚尾。")
    else:
        print("   nu 较高：厚尾存在但相对温和。")


In [ ]:
# 验证目标模型EGARCH(1,1)-t标准化残差独立同分布特征
std_resid = pd.Series(egarch_res.std_resid, index=y.index)
std_resid = std_resid.replace([np.inf, -np.inf], np.nan).dropna()
std_resid_sq = std_resid ** 2

lb_z = acorr_ljungbox(std_resid, lags=[5, 10, 20, 30], return_df=True)
lb_z2 = acorr_ljungbox(std_resid_sq, lags=[5, 10, 20, 30], return_df=True)

arch_lm_std = het_arch(std_resid, nlags=10)

egarch_diag = pd.DataFrame({
    "test": [
        "LB_z_lag5",
        "LB_z_lag10",
        "LB_z_lag20",
        "LB_z_lag30",
        "LB_z2_lag5",
        "LB_z2_lag10",
        "LB_z2_lag20",
        "LB_z2_lag30",
        "ARCH_LM_lag10"
    ],
    "p_value": [
        lb_z.loc[5, "lb_pvalue"],
        lb_z.loc[10, "lb_pvalue"],
        lb_z.loc[20, "lb_pvalue"],
        lb_z.loc[30, "lb_pvalue"],
        lb_z2.loc[5, "lb_pvalue"],
        lb_z2.loc[10, "lb_pvalue"],
        lb_z2.loc[20, "lb_pvalue"],
        lb_z2.loc[30, "lb_pvalue"],
        arch_lm_std[1]
    ]
})

egarch_diag["pass_5pct"] = egarch_diag["p_value"] > 0.05

diag_file = OUTPUT_DIR / "11_egarch11_t_residual_diagnostics.csv"
egarch_diag.to_csv(diag_file, index=False, encoding="utf-8-sig")

egarch_diag


In [ ]:
# 提取目标EGARCH模型的条件波动率时序
egarch_vol = pd.Series(egarch_res.conditional_volatility, index=y.index)
egarch_vol = egarch_vol.replace([np.inf, -np.inf], np.nan).dropna()

egarch_usage = pd.DataFrame({
    "datetime": df.loc[egarch_vol.index, "datetime"].values,
    "close": df.loc[egarch_vol.index, "close"].values,
    "ret_pct": df.loc[egarch_vol.index, "ret_pct"].values,
    "egarch_vol_pct": egarch_vol.values,
    "std_resid": std_resid.reindex(egarch_vol.index).values
})

egarch_usage["egarch_vol_pctile"] = egarch_usage["egarch_vol_pct"].rank(pct=True)

egarch_usage.head()


In [ ]:
# 基于条件波动率历史分位数划分高、中、低风险状态
def classify_vol_state(p):
    if p < 0.20:
        return "low_vol"
    elif p < 0.80:
        return "normal_vol"
    elif p < 0.95:
        return "high_vol"
    else:
        return "extreme_high_vol"

egarch_usage["vol_state"] = egarch_usage["egarch_vol_pctile"].apply(classify_vol_state)

vol_state_summary = (
    egarch_usage
    .groupby("vol_state")
    .agg(
        count=("vol_state", "count"),
        avg_vol_pct=("egarch_vol_pct", "mean"),
        median_vol_pct=("egarch_vol_pct", "median"),
        avg_abs_ret_pct=("ret_pct", lambda x: x.abs().mean()),
        max_abs_ret_pct=("ret_pct", lambda x: x.abs().max())
    )
    .reset_index()
)

usage_file = OUTPUT_DIR / "12_egarch11_t_volatility_usage_table.csv"
summary_file = OUTPUT_DIR / "13_egarch11_t_vol_state_summary.csv"

egarch_usage.to_csv(usage_file, index=False, encoding="utf-8-sig")
vol_state_summary.to_csv(summary_file, index=False, encoding="utf-8-sig")

vol_state_summary


In [ ]:
# 监测最新交易日的波动率状态区间
latest_row = egarch_usage.iloc[-1]

print("========== 最新 EGARCH 波动率状态 ==========")
print("日期：", latest_row["datetime"])
print("收盘价：", latest_row["close"])
print("当日收益率 %：", round(latest_row["ret_pct"], 4))
print("EGARCH 条件波动率 %：", round(latest_row["egarch_vol_pct"], 4))
print("历史波动率分位：", round(latest_row["egarch_vol_pctile"] * 100, 2), "%")
print("波动率状态：", latest_row["vol_state"])

if latest_row["vol_state"] == "low_vol":
    print("解释：当前处于低波动压缩区，风险表面较低，但后续可能出现波动率扩张。")
elif latest_row["vol_state"] == "normal_vol":
    print("解释：当前处于历史常态波动区，风险水平正常。")
elif latest_row["vol_state"] == "high_vol":
    print("解释：当前处于高波动区，仓位和止损需要更谨慎。")
else:
    print("解释：当前处于极端高波动区，应优先控制仓位和保证金风险。")


In [ ]:
# 绘制条件波动率与状态划分阈值线
vol_q20 = egarch_usage["egarch_vol_pct"].quantile(0.20)
vol_q80 = egarch_usage["egarch_vol_pct"].quantile(0.80)
vol_q95 = egarch_usage["egarch_vol_pct"].quantile(0.95)

plt.figure(figsize=(14, 6))
plt.plot(
    egarch_usage["datetime"],
    egarch_usage["egarch_vol_pct"],
    label="EGARCH conditional volatility",
    linewidth=1
)

plt.axhline(vol_q20, linestyle="--", linewidth=1, label="20% percentile")
plt.axhline(vol_q80, linestyle="--", linewidth=1, label="80% percentile")
plt.axhline(vol_q95, linestyle="--", linewidth=1, label="95% percentile")

plt.title("EGARCH(1,1)-t Conditional Volatility: CZCE Sugar Main Contract")
plt.xlabel("Date")
plt.ylabel("Conditional volatility (%)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 筛选并展示历史极端高波动日期区间
top_high_vol_days = (
    egarch_usage
    .sort_values("egarch_vol_pct", ascending=False)
    .head(30)
    .reset_index(drop=True)
)

top_high_vol_file = OUTPUT_DIR / "14_egarch11_t_top_high_vol_days.csv"
top_high_vol_days.to_csv(top_high_vol_file, index=False, encoding="utf-8-sig")

top_high_vol_days[
    ["datetime", "close", "ret_pct", "egarch_vol_pct", "egarch_vol_pctile", "vol_state"]
]


In [ ]:
# 基于EGARCH模型外推预测未来十日条件波动率
forecast_horizon = 10

egarch_forecast = egarch_res.forecast(
    horizon=forecast_horizon,
    method="simulation",
    simulations=5000,
    reindex=False
)

variance_forecast = egarch_forecast.variance.iloc[-1]
vol_forecast = np.sqrt(variance_forecast)

forecast_table = pd.DataFrame({
    "horizon": np.arange(1, forecast_horizon + 1),
    "variance_forecast": variance_forecast.values,
    "vol_forecast_pct": vol_forecast.values
})

forecast_table["vol_forecast_pctile_vs_history"] = forecast_table["vol_forecast_pct"].apply(
    lambda x: (egarch_usage["egarch_vol_pct"] <= x).mean()
)

forecast_file = OUTPUT_DIR / "15_egarch11_t_volatility_forecast.csv"
forecast_table.to_csv(forecast_file, index=False, encoding="utf-8-sig")

forecast_table


In [ ]:
# 绘制未来十日条件波动率预测路径图
plt.figure(figsize=(10, 5))
plt.plot(
    forecast_table["horizon"],
    forecast_table["vol_forecast_pct"],
    marker="o",
    label="Forecast volatility"
)

plt.title("EGARCH(1,1)-t Volatility Forecast")
plt.xlabel("Forecast horizon, trading days")
plt.ylabel("Forecast volatility (%)")
plt.xticks(forecast_table["horizon"])
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# 计算基于Student-t分布的单步向前多空VaR值
from scipy.stats import t as student_t

# EGARCH-t 的 nu
nu = float(egarch_res.params.get("nu", np.nan))

if np.isnan(nu):
    raise ValueError("当前模型没有 Student-t 自由度参数 nu，不能按 t 分布计算 VaR。")

# arch 包中的 t 分布是标准化 t。
std_t_scale = np.sqrt((nu - 2) / nu)

q_01 = student_t.ppf(0.01, df=nu) * std_t_scale
q_05 = student_t.ppf(0.05, df=nu) * std_t_scale
q_95 = student_t.ppf(0.95, df=nu) * std_t_scale
q_99 = student_t.ppf(0.99, df=nu) * std_t_scale

# 下一交易日预测波动率
next_vol = forecast_table.loc[forecast_table["horizon"] == 1, "vol_forecast_pct"].iloc[0]

next_mu = float(egarch_res.params.get("mu", 0.0))

next_var_table = pd.DataFrame({
    "risk_measure": [
        "Long position 95% VaR lower return",
        "Long position 99% VaR lower return",
        "Short position 95% VaR upper return",
        "Short position 99% VaR upper return"
    ],
    "return_threshold_pct": [
        next_mu + next_vol * q_05,
        next_mu + next_vol * q_01,
        next_mu + next_vol * q_95,
        next_mu + next_vol * q_99
    ]
})

var_next_file = OUTPUT_DIR / "16_egarch11_t_next_day_var.csv"
next_var_table.to_csv(var_next_file, index=False, encoding="utf-8-sig")

print("下一交易日预测均值 %：", round(next_mu, 6))
print("下一交易日预测波动率 %：", round(next_vol, 6))
print("Student-t nu：", round(nu, 6))

next_var_table


In [ ]:
# 将收益率VaR折算为价格波动上下限范围（日度变动参考）
last_close = egarch_usage["close"].iloc[-1]

price_var_table = next_var_table.copy()
price_var_table["last_close"] = last_close

price_var_table["price_threshold"] = last_close * np.exp(
    price_var_table["return_threshold_pct"] / 100
)

price_var_file = OUTPUT_DIR / "17_egarch11_t_next_day_var_price_threshold.csv"
price_var_table.to_csv(price_var_file, index=False, encoding="utf-8-sig")

price_var_table


In [ ]:
# 计算样本内单步预测VaR时序序列
var_usage = egarch_usage.copy()

var_usage["mu_pct"] = next_mu
var_usage["VaR_95_long_return_pct"] = var_usage["mu_pct"] + var_usage["egarch_vol_pct"] * q_05
var_usage["VaR_99_long_return_pct"] = var_usage["mu_pct"] + var_usage["egarch_vol_pct"] * q_01

var_usage["VaR_95_short_return_pct"] = var_usage["mu_pct"] + var_usage["egarch_vol_pct"] * q_95
var_usage["VaR_99_short_return_pct"] = var_usage["mu_pct"] + var_usage["egarch_vol_pct"] * q_99

# 多头 VaR 突破：实际收益率低于左尾阈值
var_usage["exception_95_long"] = var_usage["ret_pct"] < var_usage["VaR_95_long_return_pct"]
var_usage["exception_99_long"] = var_usage["ret_pct"] < var_usage["VaR_99_long_return_pct"]

# 空头 VaR 突破：实际收益率高于右尾阈值
var_usage["exception_95_short"] = var_usage["ret_pct"] > var_usage["VaR_95_short_return_pct"]
var_usage["exception_99_short"] = var_usage["ret_pct"] > var_usage["VaR_99_short_return_pct"]

var_usage_file = OUTPUT_DIR / "18_egarch11_t_fitted_var_series.csv"
var_usage.to_csv(var_usage_file, index=False, encoding="utf-8-sig")

var_usage.tail()


In [ ]:
# 统计样本内VaR超额穿透率（突破次数比例）
def exception_summary(series, expected_rate):
    n = len(series)
    exceptions = int(series.sum())
    rate = exceptions / n
    return {
        "n_obs": n,
        "exceptions": exceptions,
        "exception_rate": rate,
        "expected_rate": expected_rate,
        "difference": rate - expected_rate
    }

var_exception_summary = pd.DataFrame([
    {
        "side": "long",
        "confidence": "95%",
        **exception_summary(var_usage["exception_95_long"], 0.05)
    },
    {
        "side": "long",
        "confidence": "99%",
        **exception_summary(var_usage["exception_99_long"], 0.01)
    },
    {
        "side": "short",
        "confidence": "95%",
        **exception_summary(var_usage["exception_95_short"], 0.05)
    },
    {
        "side": "short",
        "confidence": "99%",
        **exception_summary(var_usage["exception_99_short"], 0.01)
    }
])

exception_file = OUTPUT_DIR / "19_egarch11_t_var_exception_summary.csv"
var_exception_summary.to_csv(exception_file, index=False, encoding="utf-8-sig")

var_exception_summary


In [ ]:
# 运行Kupiec POF覆盖率似然比检验（回测可靠性）
from scipy.stats import chi2

def kupiec_pof_test(exception_series, expected_prob):
    exception_series = pd.Series(exception_series).astype(bool)
    n = len(exception_series)
    x = int(exception_series.sum())

    if x == 0:
        # 没有突破时，似然中 x/n 为 0，单独处理
        pi_hat = 0.0
    else:
        pi_hat = x / n

    p = expected_prob

    # 避免 log(0)
    eps = 1e-12
    pi_hat_clip = min(max(pi_hat, eps), 1 - eps)
    p_clip = min(max(p, eps), 1 - eps)

    log_likelihood_null = (n - x) * np.log(1 - p_clip) + x * np.log(p_clip)
    log_likelihood_alt = (n - x) * np.log(1 - pi_hat_clip) + x * np.log(pi_hat_clip)

    lr_stat = -2 * (log_likelihood_null - log_likelihood_alt)
    p_value = 1 - chi2.cdf(lr_stat, df=1)

    return {
        "n_obs": n,
        "exceptions": x,
        "expected_prob": p,
        "actual_prob": pi_hat,
        "kupiec_lr_stat": lr_stat,
        "kupiec_p_value": p_value,
        "decision_5pct": "pass" if p_value > 0.05 else "reject"
    }

kupiec_results = pd.DataFrame([
    {
        "side": "long",
        "confidence": "95%",
        **kupiec_pof_test(var_usage["exception_95_long"], 0.05)
    },
    {
        "side": "long",
        "confidence": "99%",
        **kupiec_pof_test(var_usage["exception_99_long"], 0.01)
    },
    {
        "side": "short",
        "confidence": "95%",
        **kupiec_pof_test(var_usage["exception_95_short"], 0.05)
    },
    {
        "side": "short",
        "confidence": "99%",
        **kupiec_pof_test(var_usage["exception_99_short"], 0.01)
    }
])

kupiec_file = OUTPUT_DIR / "20_egarch11_t_kupiec_var_test.csv"
kupiec_results.to_csv(kupiec_file, index=False, encoding="utf-8-sig")

kupiec_results


In [ ]:
# 绘制收益率时序与多空VaR边界对比图
plt.figure(figsize=(14, 6))

plt.plot(
    var_usage["datetime"],
    var_usage["ret_pct"],
    linewidth=0.8,
    label="Actual return (%)"
)

plt.plot(
    var_usage["datetime"],
    var_usage["VaR_95_long_return_pct"],
    linewidth=1,
    label="Long 95% VaR"
)

plt.plot(
    var_usage["datetime"],
    var_usage["VaR_99_long_return_pct"],
    linewidth=1,
    label="Long 99% VaR"
)

plt.title("EGARCH(1,1)-t Fitted VaR: Long Position Risk")
plt.xlabel("Date")
plt.ylabel("Return (%)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 构造基于波动率目标（Volatility Targeting）的仓位缩放因子
target_vol = egarch_usage["egarch_vol_pct"].median()

position_usage = egarch_usage.copy()
position_usage["target_vol_pct"] = target_vol

position_usage["raw_position_scale"] = (
    position_usage["target_vol_pct"] / position_usage["egarch_vol_pct"]
)

# 设置仓位缩放上下限
# 0.3：最低 30% 标准仓位
# 1.5：最高 150% 标准仓位
position_usage["position_scale_capped"] = position_usage["raw_position_scale"].clip(
    lower=0.30,
    upper=1.50
)

position_file = OUTPUT_DIR / "21_egarch11_t_position_scaling.csv"
position_usage.to_csv(position_file, index=False, encoding="utf-8-sig")

position_usage.tail()


In [ ]:
# 提取并展示最新交易日的仓位缩放系数建议
latest_position = position_usage.iloc[-1]

print("========== 最新波动率仓位缩放参考 ==========")
print("日期：", latest_position["datetime"])
print("EGARCH 条件波动率 %：", round(latest_position["egarch_vol_pct"], 4))
print("目标波动率，即历史中位数 %：", round(latest_position["target_vol_pct"], 4))
print("原始仓位缩放因子：", round(latest_position["raw_position_scale"], 4))
print("截断后仓位缩放因子：", round(latest_position["position_scale_capped"], 4))
print("波动率状态：", latest_position["vol_state"])

if latest_position["position_scale_capped"] < 0.75:
    print("解释：当前波动率偏高，模型建议低于标准仓位。")
elif latest_position["position_scale_capped"] > 1.25:
    print("解释：当前波动率偏低，模型允许高于标准仓位，但仍需结合方向信号。")
else:
    print("解释：当前波动率接近常态，仓位可接近标准仓位。")


In [ ]:
# 绘制样本内仓位缩放因子时序图
plt.figure(figsize=(14, 5))

plt.plot(
    position_usage["datetime"],
    position_usage["position_scale_capped"],
    linewidth=1,
    label="Volatility-based position scale"
)

plt.axhline(1.0, linestyle="--", linewidth=1, label="Standard position")
plt.axhline(0.75, linestyle="--", linewidth=1, label="Reduced position threshold")
plt.axhline(1.25, linestyle="--", linewidth=1, label="Expanded position threshold")

plt.title("Position Scaling Based on EGARCH(1,1)-t Volatility")
plt.xlabel("Date")
plt.ylabel("Position scale")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 汇总生成EGARCH应用分析综合报告表
egarch_final_report = pd.DataFrame({
    "item": [
        "model",
        "aic",
        "bic",
        "mu",
        "omega",
        "alpha[1]",
        "gamma[1]",
        "beta[1]",
        "nu",
        "latest_date",
        "latest_close",
        "latest_return_pct",
        "latest_vol_pct",
        "latest_vol_percentile",
        "latest_vol_state",
        "next_day_forecast_vol_pct",
        "long_95_var_return_pct",
        "long_99_var_return_pct",
        "short_95_var_return_pct",
        "short_99_var_return_pct",
        "position_scale_capped"
    ],
    "value": [
        TARGET_MODEL_NAME,
        egarch_res.aic,
        egarch_res.bic,
        mu,
        omega,
        alpha1,
        gamma1,
        beta1,
        nu,
        latest_row["datetime"],
        latest_row["close"],
        latest_row["ret_pct"],
        latest_row["egarch_vol_pct"],
        latest_row["egarch_vol_pctile"],
        latest_row["vol_state"],
        next_vol,
        next_var_table.loc[0, "return_threshold_pct"],
        next_var_table.loc[1, "return_threshold_pct"],
        next_var_table.loc[2, "return_threshold_pct"],
        next_var_table.loc[3, "return_threshold_pct"],
        latest_position["position_scale_capped"]
    ]
})

final_report_file = OUTPUT_DIR / "22_egarch11_t_final_usage_report.csv"
egarch_final_report.to_csv(final_report_file, index=False, encoding="utf-8-sig")

egarch_final_report


In [ ]:
# 保存EGARCH分析与仓位管理结果至Excel
egarch_excel_file = OUTPUT_DIR / "egarch11_t_usage_results.xlsx"

with pd.ExcelWriter(egarch_excel_file, engine="openpyxl") as writer:
    egarch_param_table.to_excel(writer, sheet_name="parameters", index=False)
    egarch_diag.to_excel(writer, sheet_name="residual_diagnostics", index=False)
    egarch_usage.to_excel(writer, sheet_name="volatility_usage", index=False)
    vol_state_summary.to_excel(writer, sheet_name="vol_state_summary", index=False)
    top_high_vol_days.to_excel(writer, sheet_name="top_high_vol_days", index=False)
    forecast_table.to_excel(writer, sheet_name="vol_forecast", index=False)
    next_var_table.to_excel(writer, sheet_name="next_day_var_return", index=False)
    price_var_table.to_excel(writer, sheet_name="next_day_var_price", index=False)
    var_usage.to_excel(writer, sheet_name="fitted_var_series", index=False)
    var_exception_summary.to_excel(writer, sheet_name="var_exception_summary", index=False)
    kupiec_results.to_excel(writer, sheet_name="kupiec_test", index=False)
    position_usage.to_excel(writer, sheet_name="position_scaling", index=False)
    egarch_final_report.to_excel(writer, sheet_name="final_report", index=False)

print("EGARCH 使用结果 Excel 已保存：")
print(egarch_excel_file.resolve())


In [ ]:
# 格式化生成EGARCH波动率监控与仓位控制文字报告
print("========== EGARCH(1,1)-t 使用结论模板 ==========\n")

print(f"本文在传统 GARCH 类模型比较后，选取 {TARGET_MODEL_NAME} 作为白糖主力合约收益率的波动率刻画模型。")

print("\n1. 模型含义")
print("EGARCH(1,1)-t 同时刻画三类特征：")
print("一是波动率聚集，即大波动后更容易继续大波动；")
print("二是非对称波动，即正负冲击对未来波动率影响可能不同；")
print("三是厚尾分布，即极端收益出现概率高于正态假设。")

print("\n2. 参数解释")
if gamma_p < 0.05:
    print(f"非对称项 gamma[1] = {gamma1:.6f}，且在 5% 水平下显著，说明白糖收益率存在非对称波动结构。")
else:
    print(f"非对称项 gamma[1] = {gamma1:.6f}，但在 5% 水平下不显著，非对称波动证据不强。")

if beta_p < 0.05:
    print(f"持续性项 beta[1] = {beta1:.6f}，且显著，说明白糖波动率具有持续性。")
else:
    print(f"持续性项 beta[1] = {beta1:.6f}，但显著性不足，需要谨慎解释。")

print(f"Student-t 自由度 nu = {nu:.6f}，用于刻画收益率厚尾。")

print("\n3. 当前波动率状态")
print(f"最新日期为 {latest_row['datetime']}，EGARCH 条件波动率为 {latest_row['egarch_vol_pct']:.4f}%，")
print(f"处于历史 {latest_row['egarch_vol_pctile'] * 100:.2f}% 分位，对应状态为 {latest_row['vol_state']}。")

print("\n4. 风险预测")
print(f"下一交易日预测波动率为 {next_vol:.4f}%。")
print("基于 EGARCH-t 分布，可以构造多头和空头的 95% / 99% VaR 边界。")

print("\n5. 仓位管理")
print(f"基于历史中位数波动率作为目标波动率，当前截断后的仓位缩放因子为 {latest_position['position_scale_capped']:.4f}。")
print("该因子只用于风险调整，不提供方向信号。")

print("\n6. 模型边界")
print("EGARCH(1,1)-t 是波动率模型，不是方向预测模型。")
print("它可以服务于 VaR、止损宽度、仓位缩放和风险状态识别，但不能单独判断白糖应该做多还是做空。")
print("后续若要进一步检验模型是否仍遗漏非线性波动率结构，可接入 Hong-Lee generalized spectral test。")


In [ ]:
# 配置Hong-Lee非参数核检验的离散特征函数网格与核函数
import numpy as np
import pandas as pd
from scipy.stats import norm
from pathlib import Path
import time

# 检查前面变量是否存在
required_vars = ["egarch_res", "OUTPUT_DIR"]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} 不存在。请先运行前面 Cell 26–48。")

# 提取 EGARCH(1,1)-t 标准化残差
hl_z_series = pd.Series(egarch_res.std_resid)
hl_z_series = hl_z_series.replace([np.inf, -np.inf], np.nan).dropna()

hl_z = hl_z_series.astype(float).values
hl_T = len(hl_z)

# 不重新标准化 z_t。
# 原因：
mean_z = np.mean(hl_z)
std_z = np.std(hl_z, ddof=1)
mean_z2 = np.mean(hl_z ** 2)
mean_z4 = np.mean(hl_z ** 4)

hl_input_check = pd.DataFrame({
    "item": [
        "T",
        "mean(z)",
        "std(z)",
        "mean(z^2)",
        "mean(z^4)",
        "min(z)",
        "max(z)"
    ],
    "value": [
        hl_T,
        mean_z,
        std_z,
        mean_z2,
        mean_z4,
        np.min(hl_z),
        np.max(hl_z)
    ]
})

hl_input_file = OUTPUT_DIR / "23A_hong_lee_input_check.csv"
hl_input_check.to_csv(hl_input_file, index=False, encoding="utf-8-sig")

print("========== Hong-Lee 输入检查 ==========")
display(hl_input_check)

if hl_T < 300:
    print("警告：样本量偏短，Hong-Lee 渐近 N(0,1) 近似可能不稳定。")
else:
    print("样本量满足基本渐近检验需要。")

# 工程检查：
if not (0.90 <= mean_z2 <= 1.10):
    print("警告：mean(z^2) 明显偏离 1，请检查 EGARCH 标准化残差尺度、收益率缩放或模型估计结果。")
else:
    print("mean(z^2) 接近 1，标准化残差尺度基本合理。")


In [ ]:
# 定义Hong-Lee核检验的Bartlett核与Parzen核计算函数
def hong_lee_kernel(x, kernel="bartlett"):
    """
    Hong-Lee 检验使用的 lag kernel k(x)。

    Bartlett:
        k(x) = 1 - |x|, if |x| <= 1
             = 0,       otherwise

    Parzen:
        备用 kernel。
    """
    x = np.asarray(x, dtype=float)
    ax = np.abs(x)

    if kernel.lower() == "bartlett":
        return np.where(ax <= 1.0, 1.0 - ax, 0.0)

    elif kernel.lower() == "parzen":
        out = np.zeros_like(ax)

        mask1 = ax <= 0.5
        mask2 = (ax > 0.5) & (ax <= 1.0)

        out[mask1] = 1.0 - 6.0 * ax[mask1] ** 2 + 6.0 * ax[mask1] ** 3
        out[mask2] = 2.0 * (1.0 - ax[mask2]) ** 3

        return out

    else:
        raise ValueError("kernel 目前只支持 'bartlett' 或 'parzen'。")

def hong_lee_bandwidth(T, c=2.0, lam=0.30):
    """
    固定带宽设定：
        p = c * T^lambda

    对 bounded-support kernel，lambda 通常应小于 1/2。
    当前 lam=0.30，属于保守设定。
    """
    return float(c * (T ** lam))

HL_V_GRID = np.array(
    [-1.00, -0.75, -0.50, -0.25, 0.25, 0.50, 0.75, 1.00],
    dtype=float
)

HL_V_WEIGHTS = np.ones(len(HL_V_GRID), dtype=float) / len(HL_V_GRID)

# 可选 dense grid：
# 但 dense grid 会显著增加计算量。
HL_V_GRID_DENSE = np.array(
    [
        -1.00, -0.875, -0.75, -0.625,
        -0.50, -0.375, -0.25, -0.125,
         0.125, 0.25, 0.375, 0.50,
         0.625, 0.75, 0.875, 1.00
    ],
    dtype=float
)

HL_KERNEL = "bartlett"
HL_P_DEFAULT = hong_lee_bandwidth(hl_T, c=2.0, lam=0.30)

print("========== Hong-Lee 数值设定 ==========")
print("W0 grid:", HL_V_GRID)
print("W0 weights sum:", HL_V_WEIGHTS.sum())
print("Kernel:", HL_KERNEL)
print("Default bandwidth p:", HL_P_DEFAULT)
print("Dense W0 grid is available but not used by default.")


In [ ]:
# 实现特征函数交叉条件矩交叉乘项计算
def compute_sigma_hat_j_univariate(z, v_grid, j):
    """
    计算单变量 Hong-Lee sigma_hat_j^(2,0)(0,v)。

    Parameters
    ----------
    z : array
        标准化残差 z_t。
    v_grid : array
        transform variable v 的离散网格。
    j : int
        滞后阶数。

    Returns
    -------
    sigma_j : complex ndarray
        每个 v_grid 点上的 sigma_hat_j(v)。
    """
    z = np.asarray(z, dtype=float)
    T = len(z)

    if j <= 0 or j >= T:
        raise ValueError("j 必须满足 1 <= j <= T-1。")

    z_current = z[j:]       # z_t
    z_lagged = z[:-j]       # z_{t-j}

    eiv_lagged = np.exp(1j * np.outer(z_lagged, v_grid))
    phi_j = eiv_lagged.mean(axis=0)

    centered_eiv = eiv_lagged - phi_j
    z2_minus_1 = z_current ** 2 - 1.0

    sigma_j = -np.mean(z2_minus_1[:, None] * centered_eiv, axis=0)

    return sigma_j


In [ ]:
# 实现核心Hong-Lee核检验统计量 M(p) 与 M^o(p) 的计算（支持稳健异方差）
def hong_lee_test_univariate_fixed_bandwidth(
    z,
    p,
    v_grid=None,
    v_weights=None,
    kernel="bartlett",
    return_lag_contrib=True,
    verbose=False
):
    """
    Hong-Lee 单变量 fixed-bandwidth core M(p) / M^o(p) 检验。

    这是对单变量白糖 EGARCH 标准化残差的核心统计量实现。

    Parameters
    ----------
    z : array
        estimated standardized residuals.
    p : float
        bandwidth.
    v_grid : array
        W0(v) 的离散对称网格。
    v_weights : array
        W0(v) 离散权重。
    kernel : str
        'bartlett' or 'parzen'.
    return_lag_contrib : bool
        是否返回 lag-level Q contribution。
    verbose : bool
        是否输出 D_hat 计算进度。

    Returns
    -------
    result : dict
        Hong-Lee 主统计结果。
    lag_contrib : DataFrame
        各滞后对 Q(p) 的贡献。
    """
    start_time = time.time()

    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    T = len(z)

    if v_grid is None:
        v_grid = HL_V_GRID

    if v_weights is None:
        v_weights = HL_V_WEIGHTS

    v_grid = np.asarray(v_grid, dtype=float)
    v_weights = np.asarray(v_weights, dtype=float)
    v_weights = v_weights / v_weights.sum()

    G = len(v_grid)

    # lag kernel weights
    all_lags = np.arange(1, T)
    k_values = hong_lee_kernel(all_lags / p, kernel=kernel)
    k2_values = k_values ** 2
    k4_values = k_values ** 4

    active_mask = k2_values > 1e-14
    active_lags = all_lags[active_mask]
    active_k2 = k2_values[active_mask]

    if len(active_lags) == 0:
        raise ValueError("当前 p 太小，kernel 没有任何非零滞后。")

    eiv_full = np.exp(1j * np.outer(z, v_grid))
    phi_full = eiv_full.mean(axis=0)
    psi_full = eiv_full - phi_full

    z4_minus_1_full = z ** 4 - 1.0

    # 1. Q(p)

    Q = 0.0
    C_robust = 0.0
    lag_rows = []

    for idx, j in enumerate(active_lags):
        k2 = active_k2[idx]

        z_current = z[j:]
        z_lagged_eiv = eiv_full[:-j, :]
        n_j = T - j

        # phi_hat_j(v)
        phi_j = z_lagged_eiv.mean(axis=0)

        # sigma_hat_j(v)
        sigma_j = -np.mean(
            (z_current ** 2 - 1.0)[:, None] * (z_lagged_eiv - phi_j),
            axis=0
        )

        int_sigma_sq = np.sum(v_weights * np.abs(sigma_j) ** 2)
        q_j = k2 * n_j * int_sigma_sq
        Q += q_j

        psi_lagged = psi_full[:-j, :]
        int_abs_psi_sq = np.sum(
            v_weights * np.abs(psi_lagged) ** 2,
            axis=1
        )

        c_j = k2 * np.mean((z_current ** 4 - 1.0) * int_abs_psi_sq)
        C_robust += c_j

        if return_lag_contrib:
            lag_rows.append({
                "lag": int(j),
                "kernel_weight": float(hong_lee_kernel(j / p, kernel=kernel)),
                "k2": float(k2),
                "int_sigma_sq": float(np.real(int_sigma_sq)),
                "Q_contribution": float(np.real(q_j)),
                "C_contribution": float(np.real(c_j))
            })

    # 它对 j 和 l 做双重求和：

    D_robust = 0.0

    active_lags_D = active_lags[active_lags <= T - 2]
    active_k2_D = np.array([
        hong_lee_kernel(j / p, kernel=kernel) ** 2
        for j in active_lags_D
    ])

    for a, j in enumerate(active_lags_D):
        k2_j = active_k2_D[a]

        if verbose and a % 5 == 0:
            print(f"D robust outer loop {a+1}/{len(active_lags_D)}; lag j={j}")

        for b in range(a, len(active_lags_D)):
            l = active_lags_D[b]
            k2_l = active_k2_D[b]
            symmetry = 1.0 if a == b else 2.0

            m = l
            n_m = T - m

            weight_t = z4_minus_1_full[m:]
            psi_j_u = psi_full[m - j:T - j, :]
            psi_l_v = psi_full[m - l:T - l, :]

            A_uv = np.einsum(
                "t,tu,tv->uv",
                weight_t,
                psi_j_u,
                np.conjugate(psi_l_v),
                optimize=True
            ) / n_m

            int_abs_A_sq = np.sum(
                v_weights[:, None] * v_weights[None, :] * np.abs(A_uv) ** 2
            )

            D_robust += 2.0 * symmetry * k2_j * k2_l * int_abs_A_sq

    # 4. robust M(p)

    if D_robust <= 0 or not np.isfinite(D_robust):
        M_robust = np.nan
        pvalue_robust = np.nan
    else:
        M_robust = (Q - C_robust) / np.sqrt(D_robust)
        pvalue_robust = norm.sf(M_robust)

    # 5. iid M^o(p)
    # iid 版本使用更强假设：
    # d=1 情形：

    z4_mean_minus_1 = np.mean(z ** 4) - 1.0

    phi_v = phi_full

    sigma0_v_minus_v = 1.0 - np.abs(phi_v) ** 2
    int_sigma0_v_minus_v = np.sum(
        v_weights * np.real(sigma0_v_minus_v)
    )

    uv_sum_grid = v_grid[:, None] + v_grid[None, :]
    phi_uv = np.exp(
        1j * np.outer(z, uv_sum_grid.ravel())
    ).mean(axis=0).reshape(G, G)

    sigma0_uv = phi_uv - phi_v[:, None] * phi_v[None, :]

    int_abs_sigma0_uv_sq = np.sum(
        v_weights[:, None] * v_weights[None, :] * np.abs(sigma0_uv) ** 2
    )

    sum_k2 = np.sum(k2_values)
    sum_k4 = np.sum(k4_values[:-1])

    C_iid = z4_mean_minus_1 * int_sigma0_v_minus_v * sum_k2

    D_iid = (
        2.0
        * (z4_mean_minus_1 ** 2)
        * int_abs_sigma0_uv_sq
        * sum_k4
    )

    if D_iid <= 0 or not np.isfinite(D_iid):
        M_iid = np.nan
        pvalue_iid = np.nan
    else:
        M_iid = (Q - C_iid) / np.sqrt(D_iid)
        pvalue_iid = norm.sf(M_iid)

    elapsed = time.time() - start_time

    result = {
        "T": int(T),
        "p": float(p),
        "kernel": kernel,
        "v_grid_n": int(G),
        "v_grid_min": float(np.min(v_grid)),
        "v_grid_max": float(np.max(v_grid)),
        "max_active_lag": int(active_lags.max()),
        "active_lag_count": int(len(active_lags)),
        "Q": float(np.real(Q)),
        "C_robust": float(np.real(C_robust)),
        "D_robust": float(np.real(D_robust)),
        "M_robust": float(np.real(M_robust)) if np.isfinite(M_robust) else np.nan,
        "pvalue_robust": float(pvalue_robust) if np.isfinite(pvalue_robust) else np.nan,
        "C_iid": float(np.real(C_iid)),
        "D_iid": float(np.real(D_iid)),
        "M_iid": float(np.real(M_iid)) if np.isfinite(M_iid) else np.nan,
        "pvalue_iid": float(pvalue_iid) if np.isfinite(pvalue_iid) else np.nan,
        "elapsed_seconds": float(elapsed)
    }

    lag_contrib = pd.DataFrame(lag_rows)

    return result, lag_contrib


In [ ]:
# 运行核检验算法自检，确认非参数与参数残差诊断结果一致性
def hong_lee_implementation_self_check(z, v_grid, v_weights, p, kernel="bartlett"):
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    T = len(z)

    checks = []

    checks.append({
        "check": "sample_size",
        "value": T,
        "pass": bool(T > 300),
        "note": "T should be reasonably large for asymptotic N(0,1)."
    })

    v_sorted = np.sort(v_grid)
    symmetric = np.allclose(v_sorted, -v_sorted[::-1])

    checks.append({
        "check": "v_grid_symmetric",
        "value": str(v_grid),
        "pass": bool(symmetric),
        "note": "W0 grid should be symmetric around zero."
    })

    weights_sum = np.sum(v_weights)

    checks.append({
        "check": "v_weights_sum_to_one",
        "value": weights_sum,
        "pass": bool(np.isclose(weights_sum, 1.0)),
        "note": "Discrete W0 weights should sum to 1."
    })

    checks.append({
        "check": "bandwidth_reasonable",
        "value": p,
        "pass": bool((p > 1) and (p < T / 2)),
        "note": "p should grow with T but remain much smaller than T."
    })

    all_lags = np.arange(1, T)
    k2 = hong_lee_kernel(all_lags / p, kernel=kernel) ** 2
    active_count = int(np.sum(k2 > 1e-14))

    checks.append({
        "check": "active_lag_count",
        "value": active_count,
        "pass": bool(active_count > 0),
        "note": "At least one lag should receive nonzero kernel weight."
    })

    return pd.DataFrame(checks)

hl_self_check = hong_lee_implementation_self_check(
    z=hl_z,
    v_grid=HL_V_GRID,
    v_weights=HL_V_WEIGHTS,
    p=HL_P_DEFAULT,
    kernel=HL_KERNEL
)

hl_self_check_file = OUTPUT_DIR / "23B_hong_lee_implementation_self_check.csv"
hl_self_check.to_csv(hl_self_check_file, index=False, encoding="utf-8-sig")

hl_self_check


In [ ]:
# 在固定带宽p下运行Hong-Lee非参数检验
HL_P_MAIN = HL_P_DEFAULT

HL_P_LIST = [
    max(3.0, HL_P_MAIN * 0.50),
    HL_P_MAIN,
    HL_P_MAIN * 1.50,
    HL_P_MAIN * 2.00
]

hl_results = []
hl_lag_contrib_dict = {}

for p_val in HL_P_LIST:
    print(f"\nRunning Hong-Lee fixed-bandwidth core test: p = {p_val:.4f}")

    res, lag_contrib = hong_lee_test_univariate_fixed_bandwidth(
        z=hl_z,
        p=p_val,
        v_grid=HL_V_GRID,
        v_weights=HL_V_WEIGHTS,
        kernel=HL_KERNEL,
        return_lag_contrib=True,
        verbose=False
    )

    hl_results.append(res)
    hl_lag_contrib_dict[round(p_val, 6)] = lag_contrib

hl_results_df = pd.DataFrame(hl_results)

hl_results_df["robust_decision_5pct"] = np.where(
    hl_results_df["pvalue_robust"] < 0.05,
    "REJECT_vol_model_adequacy",
    "FAIL_TO_REJECT"
)

hl_results_df["iid_decision_5pct"] = np.where(
    hl_results_df["pvalue_iid"] < 0.05,
    "REJECT_vol_model_adequacy",
    "FAIL_TO_REJECT"
)

# 多带宽稳定性判断：
hl_results_df["reject_flag_robust"] = hl_results_df["pvalue_robust"] < 0.05
reject_share = hl_results_df["reject_flag_robust"].mean()

if reject_share >= 0.75:
    bandwidth_stability = "stable_reject"
elif reject_share <= 0.25:
    bandwidth_stability = "stable_fail_to_reject"
else:
    bandwidth_stability = "bandwidth_sensitive"

hl_results_df["bandwidth_stability"] = bandwidth_stability
hl_results_df["reject_share_across_bandwidths"] = reject_share

hl_results_file = OUTPUT_DIR / "23_hong_lee_fixed_bandwidth_core_results.csv"
hl_results_df.to_csv(hl_results_file, index=False, encoding="utf-8-sig")

print("Bandwidth stability:", bandwidth_stability)
print("Reject share across bandwidths:", reject_share)

hl_results_df


In [ ]:
# 分析各个滞后阶数对Q(p)检验统计量的贡献率
available_keys = np.array(list(hl_lag_contrib_dict.keys()), dtype=float)
closest_key = available_keys[np.argmin(np.abs(available_keys - HL_P_MAIN))]

hl_lag_contrib_main = hl_lag_contrib_dict[closest_key].copy()

if len(hl_lag_contrib_main) > 0:
    total_Q_contrib = hl_lag_contrib_main["Q_contribution"].sum()

    if total_Q_contrib > 0:
        hl_lag_contrib_main["Q_share"] = (
            hl_lag_contrib_main["Q_contribution"] / total_Q_contrib
        )
    else:
        hl_lag_contrib_main["Q_share"] = np.nan

    hl_top_lags = (
        hl_lag_contrib_main
        .sort_values("Q_contribution", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    hl_top_lags = pd.DataFrame()

hl_lag_file = OUTPUT_DIR / "24_hong_lee_main_p_lag_contributions.csv"
hl_top_lags_file = OUTPUT_DIR / "25_hong_lee_top_lag_contributions.csv"

hl_lag_contrib_main.to_csv(hl_lag_file, index=False, encoding="utf-8-sig")
hl_top_lags.to_csv(hl_top_lags_file, index=False, encoding="utf-8-sig")

print("Main p used:", closest_key)
print("Reminder: top lags are statistical Q(p) contributors, not confirmed trading cycles.")

hl_top_lags


In [ ]:
# 结合残差诊断与高阶特征检验判定EGARCH模型健康度
main_idx = np.argmin(np.abs(hl_results_df["p"] - HL_P_MAIN))
hl_main_result = hl_results_df.iloc[main_idx].copy()

hl_pvalue = float(hl_main_result["pvalue_robust"])
hl_stat = float(hl_main_result["M_robust"])

def classify_hong_lee_health(pvalue):
    if not np.isfinite(pvalue):
        return "invalid"
    elif pvalue <= 0.05:
        return "rejected"
    elif pvalue <= 0.10:
        return "borderline"
    else:
        return "healthy"

def hong_lee_health_multiplier(health):
    if health == "healthy":
        return 1.00
    elif health == "borderline":
        return 0.75
    elif health == "rejected":
        return 0.50
    else:
        return np.nan

hl_model_health = classify_hong_lee_health(hl_pvalue)
hl_health_multiplier = hong_lee_health_multiplier(hl_model_health)

hl_health_report = pd.DataFrame({
    "item": [
        "target_model",
        "hong_lee_test_scope",
        "hong_lee_test_statistic",
        "p",
        "kernel",
        "v_grid_n",
        "v_grid_min",
        "v_grid_max",
        "M_robust",
        "pvalue_robust",
        "robust_decision_5pct",
        "M_iid",
        "pvalue_iid",
        "iid_decision_5pct",
        "bandwidth_stability",
        "reject_share_across_bandwidths",
        "model_health",
        "model_health_multiplier",
        "interpretation"
    ],
    "value": [
        TARGET_MODEL_NAME if "TARGET_MODEL_NAME" in globals() else "EGARCH11_t",
        "univariate fixed-bandwidth core Hong-Lee test",
        "M(p), robust to time-varying higher-order moments",
        hl_main_result["p"],
        hl_main_result["kernel"],
        hl_main_result["v_grid_n"],
        hl_main_result["v_grid_min"],
        hl_main_result["v_grid_max"],
        hl_main_result["M_robust"],
        hl_main_result["pvalue_robust"],
        hl_main_result["robust_decision_5pct"],
        hl_main_result["M_iid"],
        hl_main_result["pvalue_iid"],
        hl_main_result["iid_decision_5pct"],
        bandwidth_stability,
        reject_share,
        hl_model_health,
        hl_health_multiplier,
        (
            "EGARCH volatility model passes Hong-Lee adequacy test"
            if hl_model_health == "healthy"
            else "EGARCH volatility model is borderline under Hong-Lee test"
            if hl_model_health == "borderline"
            else "EGARCH volatility model is rejected by Hong-Lee test"
            if hl_model_health == "rejected"
            else "Hong-Lee test invalid"
        )
    ]
})

hl_health_file = OUTPUT_DIR / "26_hong_lee_model_health_report.csv"
hl_health_report.to_csv(hl_health_file, index=False, encoding="utf-8-sig")

hl_health_report


In [ ]:
# 计算结合Hong-Lee核检验修正后的仓位缩放因子
if "position_usage" not in globals():
    raise NameError("position_usage 不存在。请先运行 Cell 43。")

hl_position_usage = position_usage.copy()

hl_position_usage["hong_lee_M_robust"] = hl_stat
hl_position_usage["hong_lee_pvalue_robust"] = hl_pvalue
hl_position_usage["hong_lee_model_health"] = hl_model_health
hl_position_usage["hong_lee_health_multiplier"] = hl_health_multiplier
hl_position_usage["hong_lee_bandwidth_stability"] = bandwidth_stability

hl_position_usage["position_scale_hong_lee_adjusted"] = (
    hl_position_usage["position_scale_capped"]
    * hl_position_usage["hong_lee_health_multiplier"]
)

# 低波动异常规则：
if hl_model_health == "rejected":
    low_vol_mask = hl_position_usage["vol_state"] == "low_vol"

    hl_position_usage.loc[
        low_vol_mask,
        "position_scale_hong_lee_adjusted"
    ] = hl_position_usage.loc[
        low_vol_mask,
        "position_scale_hong_lee_adjusted"
    ].clip(upper=1.00)

# 最终安全截断
hl_position_usage["position_scale_hong_lee_adjusted"] = (
    hl_position_usage["position_scale_hong_lee_adjusted"]
    .clip(lower=0.20, upper=1.50)
)

hl_position_file = OUTPUT_DIR / "27_position_scaling_with_hong_lee_adjustment.csv"
hl_position_usage.to_csv(hl_position_file, index=False, encoding="utf-8-sig")

hl_position_usage.tail()


In [ ]:
# 生成结合高阶核检验结论的风险状态与仓位调整文字建议
latest_hl_position = hl_position_usage.iloc[-1]

print("========== EGARCH + Hong-Lee 最新风险状态 ==========\n")

print("日期：", latest_hl_position["datetime"])
print("收盘价：", latest_hl_position["close"])
print("EGARCH 条件波动率 %：", round(latest_hl_position["egarch_vol_pct"], 6))
print("EGARCH 波动率分位：", round(latest_hl_position["egarch_vol_pctile"] * 100, 4), "%")
print("EGARCH 波动率状态：", latest_hl_position["vol_state"])

print("\n---------- Hong-Lee 模型充分性检验 ----------")
print("测试范围：univariate fixed-bandwidth core M(p)")
print("Hong-Lee M(p)：", round(hl_stat, 6))
print("Hong-Lee p-value：", round(hl_pvalue, 6))
print("Hong-Lee model health：", hl_model_health)
print("Bandwidth stability：", bandwidth_stability)
print("Reject share across bandwidths：", round(reject_share, 4))
print("Health multiplier：", hl_health_multiplier)

print("\n---------- 仓位缩放 ----------")
print("原 EGARCH 仓位缩放因子：", round(latest_hl_position["position_scale_capped"], 6))
print("Hong-Lee 调整后仓位缩放因子：", round(latest_hl_position["position_scale_hong_lee_adjusted"], 6))

print("\n---------- 解释 ----------")

if hl_model_health == "healthy":
    print("Hong-Lee 未拒绝 EGARCH(1,1)-t 的波动率模型充分性。")
    print("这说明在当前固定带宽核心检验口径下，标准化残差中未发现显著条件方差残留。")
    print("因此，EGARCH 输出的波动率分位、VaR 和仓位缩放可以正常使用。")

elif hl_model_health == "borderline":
    print("Hong-Lee 检验处于边缘区间。")
    print("这说明 EGARCH 风险模型可能存在轻微信号残留。")
    print("EGARCH 输出仍可参考，但仓位缩放应打折使用。")

elif hl_model_health == "rejected":
    print("Hong-Lee 拒绝 EGARCH(1,1)-t 的波动率模型充分性。")
    print("这意味着传统 Ljung-Box / ARCH-LM 虽然可能通过，")
    print("但标准化残差中仍存在更一般的线性或非线性条件方差残留。")
    print("此时不应机械依据低波动放大仓位，VaR 和仓位缩放需要保守化处理。")

else:
    print("Hong-Lee 检验结果无效，需要检查 D_hat(p)、W0 网格、bandwidth p 或残差输入。")

print("\n注意：Hong-Lee 不提供多空方向，只影响风险预算、仓位上限和模型可信度判断。")


In [ ]:
# 导出固定带宽Hong-Lee分析指标至Excel工作簿
hl_excel_file = OUTPUT_DIR / "hong_lee_fixed_bandwidth_core_results.xlsx"

hl_bandwidth_stability = pd.DataFrame({
    "item": [
        "bandwidth_stability",
        "reject_share_across_bandwidths",
        "bandwidth_list",
        "main_bandwidth"
    ],
    "value": [
        bandwidth_stability,
        reject_share,
        str(HL_P_LIST),
        HL_P_MAIN
    ]
})

with pd.ExcelWriter(hl_excel_file, engine="openpyxl") as writer:
    hl_input_check.to_excel(writer, sheet_name="input_check", index=False)
    hl_self_check.to_excel(writer, sheet_name="implementation_check", index=False)
    hl_results_df.to_excel(writer, sheet_name="hong_lee_results", index=False)
    hl_bandwidth_stability.to_excel(writer, sheet_name="bandwidth_stability", index=False)
    hl_health_report.to_excel(writer, sheet_name="model_health", index=False)
    hl_lag_contrib_main.to_excel(writer, sheet_name="main_p_lag_contrib", index=False)
    hl_top_lags.to_excel(writer, sheet_name="top_lag_contrib", index=False)
    hl_position_usage.to_excel(writer, sheet_name="position_adjusted", index=False)

print("Hong-Lee fixed-bandwidth core test results saved:")
print(hl_excel_file.resolve())


In [ ]:
# 格式化输出波动率健康诊断与风险控制每日总结文字报告
print("========== Hong-Lee README 段落模板 ==========\n")

print("------------------------------------------------------------")
print()
print("在传统 EGARCH(1,1)-t 模型筛选和残差诊断之后，本文进一步引入 Hong and Lee (2017) 的")
print("generalized spectral second-order derivative volatility adequacy test。")
print("该检验用于判断最终波动率模型是否仍遗漏线性或非线性的条件方差动态。")
print()
print("传统 Ljung-Box 和 ARCH-LM 检验主要检查标准化残差平方的线性自相关，")
print("而 Hong-Lee 检验进一步考察过去标准化残差信息是否仍能通过特征函数形式解释当前条件方差。")
print("因此，它可以作为 EGARCH 风险模型的二次验收工具。")
print()
print("本文在单变量白糖收益率场景下，实现 Hong-Lee 固定带宽核心检验统计量 M(p)")
print("与 i.i.d. 对照统计量 M^o(p) 的样本计算逻辑。")
print(f"主检验使用 robust M(p)，主带宽 p = {hl_main_result['p']:.4f}，kernel = {hl_main_result['kernel']}，")
print(f"W0 使用 [{hl_main_result['v_grid_min']:.2f}, {hl_main_result['v_grid_max']:.2f}] 上的对称离散网格，网格点数 = {int(hl_main_result['v_grid_n'])}。")
print()
print(f"robust M(p) = {hl_main_result['M_robust']:.6f}，")
print(f"p-value = {hl_main_result['pvalue_robust']:.6f}，")
print(f"5% 水平判断为：{hl_main_result['robust_decision_5pct']}。")
print(f"多带宽稳健性判断为：{bandwidth_stability}，")
print(f"多带宽拒绝比例为：{reject_share:.2f}。")
print()

if hl_model_health == "healthy":
    print("检验结果未拒绝 EGARCH(1,1)-t 的波动率模型充分性。")
    print("这说明在 Hong-Lee 固定带宽核心检验口径下，标准化残差中未发现显著的条件方差残留。")
    print("因此，EGARCH 输出的条件波动率、VaR 和仓位缩放可以作为主要风险输入。")

elif hl_model_health == "borderline":
    print("检验结果处于边缘区间。")
    print("这说明 EGARCH(1,1)-t 可能存在轻微的条件方差残留。")
    print("因此，EGARCH 输出仍可参考，但仓位缩放和 VaR 应保守使用。")

elif hl_model_health == "rejected":
    print("检验结果拒绝 EGARCH(1,1)-t 的波动率模型充分性。")
    print("这说明即使传统 Ljung-Box / ARCH-LM 残差诊断通过，")
    print("标准化残差中仍可能存在更一般的线性或非线性条件方差残留。")
    print("因此，本文将 Hong-Lee 结果转化为 model health flag，并对仓位缩放因子进行折扣处理。")

else:
    print("Hong-Lee 检验结果无效，需要重新检查标准化残差、bandwidth p、kernel 或 W0 网格设定。")

print()
print("交易应用上，Hong-Lee 不提供多空方向信号。")
print("它只用于判断 EGARCH 风险模型是否可信，并进一步影响 VaR 使用、仓位上限和风险预算。")
print()
print("需要注意的是，本文当前实现的是 Hong-Lee 单变量固定带宽核心检验。")
print("若要进一步完整复刻论文，还需要继续加入 data-driven bandwidth selection、Monte Carlo size/power 复现、")
print("更完整的模型对照实验以及多变量扩展。")


In [ ]:
# 配置数据驱动的最优带宽选择网格与惩罚机制
import math
import warnings
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# User controls
MC_FAST_MODE = True

if MC_FAST_MODE:
    MC_REPS = 10          # 调试用
    MC_C_GRID = [1.00, 2.00]
    MC_LAMBDA_GRID = [0.25, 0.35]
    MC_MAX_RATIO = 0.10
else:
    MC_REPS = 300
    MC_C_GRID = [1.00, 1.50, 2.00]
    MC_LAMBDA_GRID = [0.25, 0.30, 0.35]
    MC_MAX_RATIO = 0.15

MC_BURN = 500
MC_T = hl_T
MC_ALPHA = 0.05
MC_RANDOM_SEED = 20260617
MC_RESELECT_DD_P = True

np.random.seed(MC_RANDOM_SEED)

def build_bandwidth_candidates(
    T,
    c_grid=None,
    lambda_grid=None,
    min_p=3.0,
    max_ratio=0.20
):
    """
    构造候选 bandwidth 集合。

    p = c * T^lambda

    max_ratio:
        限制 p 不超过 T 的一定比例，避免有限样本中过度纳入远期 lag。
    """
    if c_grid is None:
        c_grid = [0.75, 1.00, 1.50, 2.00, 2.50, 3.00]

    if lambda_grid is None:
        lambda_grid = [0.25, 0.30, 0.35, 0.40]

    rows = []

    for c in c_grid:
        for lam in lambda_grid:
            p = c * (T ** lam)
            p = max(min_p, p)
            p = min(p, T * max_ratio)

            rows.append({
                "c": c,
                "lambda": lam,
                "p": float(p)
            })

    out = pd.DataFrame(rows)
    out["p_rounded"] = out["p"].round(4)
    out = out.drop_duplicates(subset=["p_rounded"]).reset_index(drop=True)

    return out

# 主样本用完整候选集
hl_p_candidates = build_bandwidth_candidates(
    hl_T,
    c_grid=[0.75, 1.00, 1.50, 2.00, 2.50, 3.00],
    lambda_grid=[0.25, 0.30, 0.35, 0.40],
    min_p=3.0,
    max_ratio=0.20
)

hl_p_candidates_mc = build_bandwidth_candidates(
    hl_T,
    c_grid=MC_C_GRID,
    lambda_grid=MC_LAMBDA_GRID,
    min_p=3.0,
    max_ratio=MC_MAX_RATIO
)

hl_p_candidates.to_csv(
    OUTPUT_DIR / "28_hong_lee_bandwidth_candidates_full.csv",
    index=False,
    encoding="utf-8-sig"
)

hl_p_candidates_mc.to_csv(
    OUTPUT_DIR / "28B_hong_lee_bandwidth_candidates_mc.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== Bandwidth candidate settings ==========")
print("MC_FAST_MODE:", MC_FAST_MODE)
print("MC_REPS:", MC_REPS)
print("Full candidate count:", len(hl_p_candidates))
print("MC candidate count:", len(hl_p_candidates_mc))

if MC_REPS < 100:
    print("警告：MC_REPS < 100，仅适合调试，不适合正式 size/power 统计结论。")

display(hl_p_candidates)


In [ ]:
# 运行数据驱动的最优带宽选择算法
def run_hong_lee_bandwidth_grid(
    z,
    candidate_df,
    v_grid=None,
    v_weights=None,
    kernel="bartlett",
    return_lag_contrib=False,
    label="bandwidth_grid",
    verbose=True
):
    rows = []
    lag_dict = {}

    if v_grid is None:
        v_grid = HL_V_GRID
    if v_weights is None:
        v_weights = HL_V_WEIGHTS

    for i, row in candidate_df.iterrows():
        p_val = float(row["p"])

        if verbose:
            print(f"[{label}] candidate {i+1}/{len(candidate_df)}: p = {p_val:.4f}")

        try:
            res, lag_contrib = hong_lee_test_univariate_fixed_bandwidth(
                z=z,
                p=p_val,
                v_grid=v_grid,
                v_weights=v_weights,
                kernel=kernel,
                return_lag_contrib=return_lag_contrib,
                verbose=False
            )

            res["c"] = row.get("c", np.nan)
            res["lambda"] = row.get("lambda", np.nan)
            res["status"] = "ok"
            rows.append(res)

            if return_lag_contrib:
                lag_dict[round(p_val, 6)] = lag_contrib

        except Exception as e:
            rows.append({
                "T": len(z),
                "p": p_val,
                "kernel": kernel,
                "v_grid_n": len(v_grid),
                "M_robust": np.nan,
                "pvalue_robust": np.nan,
                "M_iid": np.nan,
                "pvalue_iid": np.nan,
                "c": row.get("c", np.nan),
                "lambda": row.get("lambda", np.nan),
                "status": f"failed: {str(e)}"
            })

    out = pd.DataFrame(rows)

    out["robust_decision_5pct"] = np.where(
        out["pvalue_robust"] < 0.05,
        "REJECT_vol_model_adequacy",
        "FAIL_TO_REJECT"
    )

    return out, lag_dict

def select_project_level_bandwidth(
    results_df,
    T,
    penalty_strength=1.0
):
    """
    项目级惩罚式 bandwidth selector。

    objective(p) = M_robust(p) - penalty(p)

    注意：
    这是 project-level selector。
    selected-p p-value 是 post-selection descriptive，
    需要结合 Monte Carlo re-selection 判断其有限样本行为。
    """
    df = results_df.copy()

    df = df[
        (df["status"] == "ok")
        & np.isfinite(df["M_robust"])
        & np.isfinite(df["pvalue_robust"])
    ].copy()

    if len(df) == 0:
        raise ValueError("没有可用的 bandwidth candidate。")

    df["penalty"] = (
        penalty_strength
        * np.sqrt(df["p"] / T)
        * np.sqrt(np.log(T))
    )

    df["dd_objective"] = df["M_robust"] - df["penalty"]
    df = df.sort_values("dd_objective", ascending=False).reset_index(drop=True)

    selected = df.iloc[0].copy()

    return selected, df


In [ ]:
# 对比固定带宽与最优数据驱动带宽下的检验统计量结论
hl_dd_results_df, hl_dd_lag_contrib = run_hong_lee_bandwidth_grid(
    z=hl_z,
    candidate_df=hl_p_candidates,
    v_grid=HL_V_GRID,
    v_weights=HL_V_WEIGHTS,
    kernel=HL_KERNEL,
    return_lag_contrib=False,
    label="main_project_dd",
    verbose=True
)

hl_dd_selected, hl_dd_ranked = select_project_level_bandwidth(
    results_df=hl_dd_results_df,
    T=hl_T,
    penalty_strength=1.0
)

hl_dd_results_df.to_csv(
    OUTPUT_DIR / "29_hong_lee_project_level_bandwidth_results.csv",
    index=False,
    encoding="utf-8-sig"
)

hl_dd_ranked.to_csv(
    OUTPUT_DIR / "30_hong_lee_project_level_bandwidth_ranked.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== Project-level data-driven bandwidth selected ==========")
print("Selected p:", hl_dd_selected["p"])
print("M_robust:", hl_dd_selected["M_robust"])
print("p-value:", hl_dd_selected["pvalue_robust"])
print("Decision:", hl_dd_selected["robust_decision_5pct"])
print("Objective:", hl_dd_selected["dd_objective"])
print("Boundary: selected-p p-value is post-selection descriptive.")

hl_dd_ranked.head(10)


In [ ]:
# 采用最优数据驱动带宽更新模型设定健康度评估
fixed_main_decision = hl_main_result["robust_decision_5pct"]
sample_selected_p_decision = hl_dd_selected["robust_decision_5pct"]

if fixed_main_decision == sample_selected_p_decision and bandwidth_stability in ["stable_reject", "stable_fail_to_reject"]:
    final_bandwidth_conclusion = "bandwidth_robust"
elif fixed_main_decision == sample_selected_p_decision:
    final_bandwidth_conclusion = "fixed_and_sample_selected_p_agree_but_multi_p_sensitive"
else:
    final_bandwidth_conclusion = "fixed_and_sample_selected_p_conflict"

hl_bandwidth_comparison = pd.DataFrame({
    "item": [
        "fixed_main_p",
        "fixed_main_M",
        "fixed_main_pvalue",
        "fixed_main_decision",
        "multi_p_stability",
        "multi_p_reject_share",
        "sample_selected_p",
        "sample_selected_p_M",
        "sample_selected_p_pvalue",
        "sample_selected_p_decision",
        "final_bandwidth_conclusion",
        "post_selection_boundary",
        "important_boundary"
    ],
    "value": [
        HL_P_MAIN,
        hl_main_result["M_robust"],
        hl_main_result["pvalue_robust"],
        fixed_main_decision,
        bandwidth_stability,
        reject_share,
        hl_dd_selected["p"],
        hl_dd_selected["M_robust"],
        hl_dd_selected["pvalue_robust"],
        sample_selected_p_decision,
        final_bandwidth_conclusion,
        "selected-p p-value is descriptive; finite-sample behavior is assessed by MC re-selection",
        "project-level penalized selector, not paper Section 6 exact plug-in selector"
    ]
})

hl_bandwidth_comparison.to_csv(
    OUTPUT_DIR / "31_hong_lee_bandwidth_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

hl_bandwidth_comparison


In [ ]:
# 更新最优带宽下的头寸缩放因子
sample_selected_pvalue = float(hl_dd_selected["pvalue_robust"])
sample_selected_health = classify_hong_lee_health(sample_selected_pvalue)

if final_bandwidth_conclusion == "bandwidth_robust":
    if sample_selected_p_decision == "REJECT_vol_model_adequacy":
        hl_final_model_health = "rejected"
    else:
        hl_final_model_health = "healthy"
else:
    hl_final_model_health = "borderline"

hl_final_health_multiplier = hong_lee_health_multiplier(hl_final_model_health)

hl_final_health_report = pd.DataFrame({
    "item": [
        "fixed_p_health",
        "sample_selected_p_health",
        "bandwidth_conclusion",
        "final_model_health",
        "final_health_multiplier",
        "boundary"
    ],
    "value": [
        hl_model_health,
        sample_selected_health,
        final_bandwidth_conclusion,
        hl_final_model_health,
        hl_final_health_multiplier,
        "model health is application-layer risk flag, not original paper classification"
    ]
})

hl_final_health_report.to_csv(
    OUTPUT_DIR / "32_hong_lee_final_model_health_after_sample_selected_p.csv",
    index=False,
    encoding="utf-8-sig"
)

hl_final_health_report


In [ ]:
# 汇总并保存最优带宽选择结果至Excel
hl_position_usage_dd = hl_position_usage.copy()

hl_position_usage_dd["hong_lee_sample_selected_p"] = float(hl_dd_selected["p"])
hl_position_usage_dd["hong_lee_sample_selected_p_M"] = float(hl_dd_selected["M_robust"])
hl_position_usage_dd["hong_lee_sample_selected_p_pvalue"] = float(hl_dd_selected["pvalue_robust"])
hl_position_usage_dd["hong_lee_final_model_health"] = hl_final_model_health
hl_position_usage_dd["hong_lee_final_health_multiplier"] = hl_final_health_multiplier
hl_position_usage_dd["hong_lee_final_bandwidth_conclusion"] = final_bandwidth_conclusion

hl_position_usage_dd["position_scale_hong_lee_final"] = (
    hl_position_usage_dd["position_scale_capped"]
    * hl_position_usage_dd["hong_lee_final_health_multiplier"]
)

if hl_final_model_health == "rejected":
    low_vol_mask = hl_position_usage_dd["vol_state"] == "low_vol"
    hl_position_usage_dd.loc[
        low_vol_mask,
        "position_scale_hong_lee_final"
    ] = hl_position_usage_dd.loc[
        low_vol_mask,
        "position_scale_hong_lee_final"
    ].clip(upper=1.00)

hl_position_usage_dd["position_scale_hong_lee_final"] = (
    hl_position_usage_dd["position_scale_hong_lee_final"]
    .clip(lower=0.20, upper=1.50)
)

hl_position_usage_dd.to_csv(
    OUTPUT_DIR / "33_position_scaling_after_sample_selected_bandwidth.csv",
    index=False,
    encoding="utf-8-sig"
)

hl_position_usage_dd.tail()


In [ ]:
# 蒙特卡洛模拟辅助分析：原假设（H0）下 size 显著性检验参数配置
def binom_rate_ci(k, n, z=1.96):
    """
    Wilson score interval for binomial rejection rate.

    相比 Wald interval：
    - k = 0 时不会返回 [0, 0]
    - k = n 时不会返回 [1, 1]
    - 小样本下更稳健

    返回：
        p_hat, ci_low, ci_high
    """
    if n <= 0:
        return np.nan, np.nan, np.nan

    k = int(k)
    n = int(n)

    if k < 0 or k > n:
        return np.nan, np.nan, np.nan

    p_hat = k / n

    denom = 1.0 + (z ** 2) / n

    center = (
        p_hat
        + (z ** 2) / (2.0 * n)
    ) / denom

    half_width = (
        z
        * np.sqrt(
            (p_hat * (1.0 - p_hat) / n)
            + (z ** 2) / (4.0 * n ** 2)
        )
        / denom
    )

    ci_low = max(0.0, center - half_width)
    ci_high = min(1.0, center + half_width)

    return p_hat, ci_low, ci_high

def summarize_mc_rejection(df, reject_col, label, alpha=0.05):
    """
    稳健版 MC 拒绝率汇总函数。

    可处理：
    1. reject_col 不存在；
    2. 全部失败；
    3. reject_col 全部为空；
    4. k=0 或 k=n 时 CI 不退化；
    5. 显式标注 CI 方法为 Wilson。
    """
    if reject_col not in df.columns:
        return {
            "label": label,
            "nominal_alpha": alpha,
            "n_success": 0,
            "reject_count": 0,
            "rejection_rate": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "ci_method": "Wilson",
            "status": f"missing_column: {reject_col}"
        }

    if "status" not in df.columns:
        return {
            "label": label,
            "nominal_alpha": alpha,
            "n_success": 0,
            "reject_count": 0,
            "rejection_rate": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "ci_method": "Wilson",
            "status": "missing_status_column"
        }

    ok = df[(df["status"] == "ok") & df[reject_col].notna()].copy()
    n = len(ok)

    if n > 0:
        k = int(ok[reject_col].astype(bool).sum())
        rate, ci_lo, ci_hi = binom_rate_ci(k, n)
        status = "ok"
    else:
        k, rate, ci_lo, ci_hi = 0, np.nan, np.nan, np.nan
        status = "no_successful_replications"

    return {
        "label": label,
        "nominal_alpha": alpha,
        "n_success": n,
        "reject_count": k,
        "rejection_rate": rate,
        "ci95_low": ci_lo,
        "ci95_high": ci_hi,
        "ci_method": "Wilson",
        "status": status
    }

print("========== Monte Carlo settings ==========")
print("MC_FAST_MODE:", MC_FAST_MODE)
print("MC_REPS:", MC_REPS)
print("MC_T:", MC_T)
print("MC_BURN:", MC_BURN)
print("MC_ALPHA:", MC_ALPHA)
print("MC_RESELECT_DD_P:", MC_RESELECT_DD_P)

if MC_REPS < 100:
    print("警告：MC_REPS < 100，仅用于代码调试，不用于正式统计结论。")

# sanity check
_test_rate, _test_low, _test_high = binom_rate_ci(0, 10)
print("Wilson CI sanity check for 0/10:")
print("rate =", _test_rate, "ci95_low =", _test_low, "ci95_high =", _test_high)


In [ ]:
# 蒙特卡洛模拟辅助拟合函数（自动估计EGARCH模型）
def refit_egarch11_t(y_sim):
    """
    对模拟收益率重新估计 EGARCH(1,1)-t。
    """
    am = arch_model(
        y_sim,
        mean="Constant",
        vol="EGARCH",
        p=1,
        o=1,
        q=1,
        dist="t",
        rescale=False
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = am.fit(disp="off")

    return fit

def standardized_residual_checks(fit):
    """
    记录重新估计模型的收敛与标准化残差尺度。
    """
    z = pd.Series(fit.std_resid).replace([np.inf, -np.inf], np.nan).dropna()

    return {
        "convergence_flag": getattr(fit, "convergence_flag", np.nan),
        "loglikelihood": getattr(fit, "loglikelihood", np.nan),
        "aic": getattr(fit, "aic", np.nan),
        "bic": getattr(fit, "bic", np.nan),
        "std_resid_n": len(z),
        "std_resid_mean": float(z.mean()) if len(z) > 0 else np.nan,
        "std_resid_mean_sq": float((z ** 2).mean()) if len(z) > 0 else np.nan
    }

def traditional_residual_tests(fit, lags=10):
    """
    传统对照检验：
    1. Ljung-Box(z)：检查标准化残差方向性线性自相关；
    2. Ljung-Box(z²)：检查标准化残差平方自相关；
    3. ARCH-LM：检查 ARCH 残留。
    """
    z = pd.Series(fit.std_resid).replace([np.inf, -np.inf], np.nan).dropna()

    try:
        lb_z = acorr_ljungbox(z, lags=[lags], return_df=True)
        lb_z_p = float(lb_z["lb_pvalue"].iloc[0])
    except Exception:
        lb_z_p = np.nan

    try:
        lb_z2 = acorr_ljungbox(z ** 2, lags=[lags], return_df=True)
        lb_z2_p = float(lb_z2["lb_pvalue"].iloc[0])
    except Exception:
        lb_z2_p = np.nan

    try:
        arch_lm = het_arch(z, nlags=lags)
        arch_lm_p = float(arch_lm[1])
    except Exception:
        arch_lm_p = np.nan

    return {
        "lb_z_lag10_pvalue": lb_z_p,
        "lb_z2_lag10_pvalue": lb_z2_p,
        "arch_lm_lag10_pvalue": arch_lm_p,
        "lb_z_reject_5pct": lb_z_p < 0.05 if np.isfinite(lb_z_p) else np.nan,
        "lb_z2_reject_5pct": lb_z2_p < 0.05 if np.isfinite(lb_z2_p) else np.nan,
        "arch_lm_reject_5pct": arch_lm_p < 0.05 if np.isfinite(arch_lm_p) else np.nan
    }

def run_hong_lee_sample_selected_and_dd_on_fit(
    fit,
    sample_selected_p,
    candidate_df_mc,
    label="mc"
):
    """
    每次 MC 复制中：
    1. 使用真实白糖样本上选出的 sample_selected_p，固定用于该复制；
    2. 在该模拟样本内部重新执行 project-level data-driven p selection；
    3. 同时记录 Ljung-Box(z)、Ljung-Box(z²)、ARCH-LM。

    注意：
    sample_selected_p 不是最早的固定主带宽 HL_P_MAIN，
    它是从真实白糖样本中选出的 project-level selected p。
    """
    z_mc = pd.Series(fit.std_resid).replace([np.inf, -np.inf], np.nan).dropna().values

    sample_res, _ = hong_lee_test_univariate_fixed_bandwidth(
        z=z_mc,
        p=sample_selected_p,
        v_grid=HL_V_GRID,
        v_weights=HL_V_WEIGHTS,
        kernel=HL_KERNEL,
        return_lag_contrib=False,
        verbose=False
    )

    sample_selected_res = {
        "hl_sample_selected_p": sample_selected_p,
        "hl_sample_selected_p_M": sample_res["M_robust"],
        "hl_sample_selected_p_pvalue": sample_res["pvalue_robust"],
        "hl_sample_selected_p_reject_5pct": sample_res["pvalue_robust"] < MC_ALPHA
    }

    if MC_RESELECT_DD_P:
        dd_grid_results, _ = run_hong_lee_bandwidth_grid(
            z=z_mc,
            candidate_df=candidate_df_mc,
            v_grid=HL_V_GRID,
            v_weights=HL_V_WEIGHTS,
            kernel=HL_KERNEL,
            return_lag_contrib=False,
            label=f"{label}_dd",
            verbose=False
        )

        dd_selected, dd_ranked = select_project_level_bandwidth(
            results_df=dd_grid_results,
            T=len(z_mc),
            penalty_strength=1.0
        )

        dd_res = {
            "hl_reselected_dd_p": float(dd_selected["p"]),
            "hl_reselected_dd_M": float(dd_selected["M_robust"]),
            "hl_reselected_dd_pvalue": float(dd_selected["pvalue_robust"]),
            "hl_reselected_dd_reject_5pct": bool(dd_selected["pvalue_robust"] < MC_ALPHA)
        }
    else:
        dd_res = {
            "hl_reselected_dd_p": np.nan,
            "hl_reselected_dd_M": np.nan,
            "hl_reselected_dd_pvalue": np.nan,
            "hl_reselected_dd_reject_5pct": np.nan
        }

    out = {}
    out.update(sample_selected_res)
    out.update(dd_res)
    out.update(standardized_residual_checks(fit))
    out.update(traditional_residual_tests(fit, lags=10))

    return out


In [ ]:
# 蒙特卡洛H0模拟数据生成过程DGP（EGARCH-t模拟）
def simulate_from_fitted_egarch(res, nobs, burn=500, random_seed=None):
    """
    从 fitted EGARCH(1,1)-t 生成模拟收益率。
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    sim = res.model.simulate(
        params=res.params,
        nobs=nobs,
        burn=burn
    )

    y_sim = pd.Series(sim["data"]).replace([np.inf, -np.inf], np.nan).dropna()

    return y_sim


In [ ]:
# 执行原假设下的蒙特卡洛第一类错误（Size）模拟循环
mc_size_rows = []

for r in range(MC_REPS):
    print(f"\nMC size replication {r+1}/{MC_REPS}")

    try:
        y_sim = simulate_from_fitted_egarch(
            res=egarch_res,
            nobs=MC_T,
            burn=MC_BURN,
            random_seed=MC_RANDOM_SEED + r
        )

        fit_sim = refit_egarch11_t(y_sim)

        row = run_hong_lee_sample_selected_and_dd_on_fit(
            fit=fit_sim,
            sample_selected_p=float(hl_dd_selected["p"]),
            candidate_df_mc=hl_p_candidates_mc,
            label=f"size_{r+1}"
        )

        row["rep"] = r + 1
        row["mc_type"] = "size_H0_fitted_EGARCH"
        row["status"] = "ok"

        mc_size_rows.append(row)

    except Exception as e:
        mc_size_rows.append({
            "rep": r + 1,
            "mc_type": "size_H0_fitted_EGARCH",
            "status": f"failed: {str(e)}"
        })

mc_size_results = pd.DataFrame(mc_size_rows)

mc_size_results.to_csv(
    OUTPUT_DIR / "34_monte_carlo_size_results.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_size_results.tail()


In [ ]:
# 整理与对比原假设下核检验的拒绝概率（实际Size）
size_sample_selected_summary = summarize_mc_rejection(
    mc_size_results,
    "hl_sample_selected_p_reject_5pct",
    "size_sample_selected_p",
    alpha=MC_ALPHA
)

size_reselected_dd_summary = summarize_mc_rejection(
    mc_size_results,
    "hl_reselected_dd_reject_5pct",
    "size_reselected_project_dd_p",
    alpha=MC_ALPHA
)

size_lb_z_summary = summarize_mc_rejection(
    mc_size_results,
    "lb_z_reject_5pct",
    "size_ljungbox_z",
    alpha=MC_ALPHA
)

size_lb_z2_summary = summarize_mc_rejection(
    mc_size_results,
    "lb_z2_reject_5pct",
    "size_ljungbox_z2",
    alpha=MC_ALPHA
)

size_arch_summary = summarize_mc_rejection(
    mc_size_results,
    "arch_lm_reject_5pct",
    "size_arch_lm",
    alpha=MC_ALPHA
)

mc_size_summary = pd.DataFrame([
    size_sample_selected_summary,
    size_reselected_dd_summary,
    size_lb_z_summary,
    size_lb_z2_summary,
    size_arch_summary
])

def interpret_size_row(row, alpha=0.05, mc_reps=MC_REPS):
    """
    Size interpretation.

    MC_REPS < 100:
        只标记 debugging_only，不做 too_conservative / too_aggressive 判断。

    MC_REPS >= 100:
        才根据 rejection_rate 与 nominal alpha 的偏离做初步判断。
    """
    if row.get("status") != "ok":
        return "invalid"

    rate = row.get("rejection_rate", np.nan)

    if not np.isfinite(rate):
        return "invalid"

    if mc_reps < 100:
        return "debugging_only"

    if rate > alpha + 0.05:
        return "too_aggressive"

    if rate < max(0.0, alpha - 0.04):
        return "too_conservative"

    return "roughly_acceptable"

mc_size_summary["interpretation"] = mc_size_summary.apply(
    lambda row: interpret_size_row(row, alpha=MC_ALPHA, mc_reps=MC_REPS),
    axis=1
)

mc_size_summary["mc_boundary"] = np.where(
    MC_REPS < 100,
    "debugging_only_MC_REPS_below_100",
    "usable_for_preliminary_inference"
)

mc_size_summary.to_csv(
    OUTPUT_DIR / "35_monte_carlo_size_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_size_summary


In [ ]:
# 蒙特卡洛模拟第二部分：备择假设（H1）下检验功效（Power）校准与配置
def infer_model_y_std(default_value=0.84):
    """
    尽量从 egarch_res 中推断原始收益率序列的日波动率尺度。
    如果失败，则使用默认 0.84。
    """
    candidates = []

    for attr in ["_y", "y"]:
        try:
            y = getattr(egarch_res.model, attr)
            y = pd.Series(y).replace([np.inf, -np.inf], np.nan).dropna()
            if len(y) > 20:
                candidates.append(float(y.std()))
        except Exception:
            pass

    if len(candidates) > 0 and np.isfinite(candidates[0]) and candidates[0] > 0:
        return candidates[0]

    return default_value

SR_TARGET_DAILY_VOL = infer_model_y_std(default_value=0.84)

def gjr_uncond_vol(omega, alpha, gamma, beta):
    denom = 1.0 - alpha - 0.5 * gamma - beta
    if denom <= 0:
        return np.nan
    return float(np.sqrt(omega / denom))

def calibrate_gjr_omega(target_vol, alpha, gamma, beta):
    denom = 1.0 - alpha - 0.5 * gamma - beta
    if denom <= 0:
        raise ValueError("GJR parameters are not covariance-stationary under the approximation.")
    return float((target_vol ** 2) * denom)

print("Inferred SR target daily volatility scale:", SR_TARGET_DAILY_VOL)


In [ ]:
# 备择假设H1：非对称GJR-GARCH-t模拟数据生成过程DGP
def simulate_gjr_t_dgp(
    nobs,
    burn=500,
    omega=0.02,
    alpha=0.05,
    gamma=0.10,
    beta=0.88,
    nu=6.0,
    mu=0.0,
    random_seed=None
):
    """
    显式 GJR-GARCH-t DGP。
    输出单位与原始收益率序列尺度类似。
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    total_n = nobs + burn

    raw = np.random.standard_t(df=nu, size=total_n)
    z = raw * np.sqrt((nu - 2.0) / nu)

    eps = np.zeros(total_n)
    h = np.zeros(total_n)

    denom = 1.0 - alpha - 0.5 * gamma - beta
    if denom > 0:
        h0 = omega / denom
    else:
        h0 = omega / max(1e-6, 1.0 - beta)

    h[0] = max(h0, 1e-8)
    eps[0] = np.sqrt(h[0]) * z[0]

    for t in range(1, total_n):
        indicator = 1.0 if eps[t - 1] < 0 else 0.0

        h[t] = (
            omega
            + alpha * eps[t - 1] ** 2
            + gamma * indicator * eps[t - 1] ** 2
            + beta * h[t - 1]
        )

        h[t] = max(h[t], 1e-10)
        eps[t] = np.sqrt(h[t]) * z[t]

    y = mu + eps

    return pd.Series(y[burn:]).reset_index(drop=True)


In [ ]:
# 备择假设H1：机制转换（Regime-Switching）波动率模拟数据生成过程DGP
def simulate_regime_switching_vol_dgp(
    nobs,
    burn=500,
    sigma_low=None,
    sigma_high=None,
    p_stay_low=0.97,
    p_stay_high=0.92,
    nu=6.0,
    mu=0.0,
    random_seed=None
):
    """
    简单 regime-switching volatility DGP。
    输出单位与原始收益率序列尺度类似。
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    if sigma_low is None:
        sigma_low = 0.65 * SR_TARGET_DAILY_VOL

    if sigma_high is None:
        sigma_high = 1.90 * SR_TARGET_DAILY_VOL

    total_n = nobs + burn

    states = np.zeros(total_n, dtype=int)

    raw = np.random.standard_t(df=nu, size=total_n)
    z = raw * np.sqrt((nu - 2.0) / nu)

    for t in range(1, total_n):
        if states[t - 1] == 0:
            states[t] = 0 if np.random.rand() < p_stay_low else 1
        else:
            states[t] = 1 if np.random.rand() < p_stay_high else 0

    sigma = np.where(states == 0, sigma_low, sigma_high)
    y = mu + sigma * z

    return pd.Series(y[burn:]).reset_index(drop=True)


In [ ]:
# 设定备择假设（H1）下MC模拟的非对称与机制转换波动率参数组
GJR_MODERATE = {
    "alpha": 0.05,
    "gamma": 0.08,
    "beta": 0.88,
    "nu": 6.0
}

GJR_STRONG = {
    "alpha": 0.05,
    "gamma": 0.15,
    "beta": 0.84,
    "nu": 6.0
}

GJR_MODERATE["omega"] = calibrate_gjr_omega(
    target_vol=SR_TARGET_DAILY_VOL,
    alpha=GJR_MODERATE["alpha"],
    gamma=GJR_MODERATE["gamma"],
    beta=GJR_MODERATE["beta"]
)

GJR_STRONG["omega"] = calibrate_gjr_omega(
    target_vol=SR_TARGET_DAILY_VOL,
    alpha=GJR_STRONG["alpha"],
    gamma=GJR_STRONG["gamma"],
    beta=GJR_STRONG["beta"]
)

POWER_DGP_CONFIGS = [
    {
        "dgp_name": "gjr_threshold_moderate",
        "dgp_type": "gjr_t",
        **GJR_MODERATE
    },
    {
        "dgp_name": "gjr_threshold_strong",
        "dgp_type": "gjr_t",
        **GJR_STRONG
    },
    {
        "dgp_name": "regime_switching_vol",
        "dgp_type": "regime_switching",
        "sigma_low": 0.65 * SR_TARGET_DAILY_VOL,
        "sigma_high": 1.90 * SR_TARGET_DAILY_VOL,
        "p_stay_low": 0.97,
        "p_stay_high": 0.92,
        "nu": 6.0
    }
]

power_dgp_config_df = pd.DataFrame(POWER_DGP_CONFIGS)

# 增加 GJR 无条件波动率检查
power_dgp_config_df["implied_uncond_vol"] = np.nan

for i, row in power_dgp_config_df.iterrows():
    if row["dgp_type"] == "gjr_t":
        power_dgp_config_df.loc[i, "implied_uncond_vol"] = gjr_uncond_vol(
            omega=row["omega"],
            alpha=row["alpha"],
            gamma=row["gamma"],
            beta=row["beta"]
        )

power_dgp_config_df["target_sr_vol"] = SR_TARGET_DAILY_VOL

power_dgp_config_df.to_csv(
    OUTPUT_DIR / "36A_power_dgp_configurations.csv",
    index=False,
    encoding="utf-8-sig"
)

power_dgp_config_df


In [ ]:
# 执行备择假设下的蒙特卡洛拒绝率（Power）模拟循环
mc_power_rows = []

for cfg_idx, cfg in enumerate(POWER_DGP_CONFIGS):
    dgp_name = cfg["dgp_name"]
    dgp_type = cfg["dgp_type"]

    for r in range(MC_REPS):
        print(f"\nMC power {dgp_name}: replication {r+1}/{MC_REPS}")

        try:
            seed = MC_RANDOM_SEED + 10000 * (cfg_idx + 1) + r

            if dgp_type == "gjr_t":
                y_alt = simulate_gjr_t_dgp(
                    nobs=MC_T,
                    burn=MC_BURN,
                    omega=cfg["omega"],
                    alpha=cfg["alpha"],
                    gamma=cfg["gamma"],
                    beta=cfg["beta"],
                    nu=cfg["nu"],
                    mu=0.0,
                    random_seed=seed
                )

            elif dgp_type == "regime_switching":
                y_alt = simulate_regime_switching_vol_dgp(
                    nobs=MC_T,
                    burn=MC_BURN,
                    sigma_low=cfg["sigma_low"],
                    sigma_high=cfg["sigma_high"],
                    p_stay_low=cfg["p_stay_low"],
                    p_stay_high=cfg["p_stay_high"],
                    nu=cfg["nu"],
                    mu=0.0,
                    random_seed=seed
                )

            else:
                raise ValueError(f"Unknown DGP type: {dgp_type}")

            fit_alt = refit_egarch11_t(y_alt)

            row = run_hong_lee_sample_selected_and_dd_on_fit(
                fit=fit_alt,
                sample_selected_p=float(hl_dd_selected["p"]),
                candidate_df_mc=hl_p_candidates_mc,
                label=f"power_{dgp_name}_{r+1}"
            )

            row["rep"] = r + 1
            row["mc_type"] = "power_H1"
            row["dgp_name"] = dgp_name
            row["dgp_type"] = dgp_type
            row["status"] = "ok"

            mc_power_rows.append(row)

        except Exception as e:
            mc_power_rows.append({
                "rep": r + 1,
                "mc_type": "power_H1",
                "dgp_name": dgp_name,
                "dgp_type": dgp_type,
                "status": f"failed: {str(e)}"
            })

mc_power_results = pd.DataFrame(mc_power_rows)

mc_power_results.to_csv(
    OUTPUT_DIR / "36_monte_carlo_power_results.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_power_results.tail()


In [ ]:
# 整理并汇总各备择假设下核检验功效（Power）拒绝率
power_summary_rows = []

for dgp_name, group in mc_power_results.groupby("dgp_name"):
    power_summary_rows.append(
        summarize_mc_rejection(
            group,
            "hl_sample_selected_p_reject_5pct",
            f"power_sample_selected_p_{dgp_name}",
            alpha=MC_ALPHA
        )
    )

    power_summary_rows.append(
        summarize_mc_rejection(
            group,
            "hl_reselected_dd_reject_5pct",
            f"power_reselected_project_dd_p_{dgp_name}",
            alpha=MC_ALPHA
        )
    )

    power_summary_rows.append(
        summarize_mc_rejection(
            group,
            "lb_z_reject_5pct",
            f"power_ljungbox_z_{dgp_name}",
            alpha=MC_ALPHA
        )
    )

    power_summary_rows.append(
        summarize_mc_rejection(
            group,
            "lb_z2_reject_5pct",
            f"power_ljungbox_z2_{dgp_name}",
            alpha=MC_ALPHA
        )
    )

    power_summary_rows.append(
        summarize_mc_rejection(
            group,
            "arch_lm_reject_5pct",
            f"power_arch_lm_{dgp_name}",
            alpha=MC_ALPHA
        )
    )

mc_power_summary = pd.DataFrame(power_summary_rows)

def interpret_power_row(row, mc_reps=MC_REPS):
    """
    Power interpretation.

    MC_REPS < 100:
        只标记 debugging_only，不做 low/moderate/high power 判断。

    MC_REPS >= 100:
        才根据 rejection_rate 做初步 power 判断。
    """
    if row.get("status") != "ok":
        return "invalid"

    rate = row.get("rejection_rate", np.nan)

    if not np.isfinite(rate):
        return "invalid"

    if mc_reps < 100:
        return "debugging_only"

    if rate >= 0.70:
        return "high_power"

    if rate >= 0.40:
        return "moderate_power"

    return "low_power"

mc_power_summary["power_interpretation"] = mc_power_summary.apply(
    lambda row: interpret_power_row(row, mc_reps=MC_REPS),
    axis=1
)

mc_power_summary["mc_boundary"] = np.where(
    MC_REPS < 100,
    "debugging_only_MC_REPS_below_100",
    "usable_for_preliminary_inference"
)

mc_power_summary.to_csv(
    OUTPUT_DIR / "37_monte_carlo_power_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_power_summary


In [ ]:
# 综合展示与对比Size与Power拒绝率权衡表
mc_validation_summary = pd.concat(
    [
        mc_size_summary.assign(section="size"),
        mc_power_summary.assign(section="power")
    ],
    axis=0,
    ignore_index=True
)

mc_validation_summary.to_csv(
    OUTPUT_DIR / "38_monte_carlo_validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

size_dd_row = mc_size_summary[
    mc_size_summary["label"] == "size_reselected_project_dd_p"
]

if len(size_dd_row) > 0:
    empirical_size_dd = float(size_dd_row["rejection_rate"].iloc[0])
    empirical_size_dd_ci_low = float(size_dd_row["ci95_low"].iloc[0])
    empirical_size_dd_ci_high = float(size_dd_row["ci95_high"].iloc[0])
else:
    empirical_size_dd = np.nan
    empirical_size_dd_ci_low = np.nan
    empirical_size_dd_ci_high = np.nan

power_dd_rows = mc_power_summary[
    mc_power_summary["label"].str.startswith("power_reselected_project_dd_p_")
].copy()

if len(power_dd_rows) > 0:
    avg_power_dd = float(power_dd_rows["rejection_rate"].mean())
else:
    avg_power_dd = np.nan

if MC_REPS < 100:
    mc_reliability = "debugging_only"
elif not np.isfinite(empirical_size_dd) or not np.isfinite(avg_power_dd):
    mc_reliability = "invalid"
elif empirical_size_dd > MC_ALPHA + 0.05:
    mc_reliability = "size_too_aggressive"
elif empirical_size_dd < max(0.0, MC_ALPHA - 0.04):
    mc_reliability = "size_too_conservative"
elif avg_power_dd < 0.40:
    mc_reliability = "low_power"
else:
    mc_reliability = "acceptable"

hl_mc_reliability_report = pd.DataFrame({
    "item": [
        "empirical_size_reselected_project_dd",
        "empirical_size_reselected_project_dd_ci95_low",
        "empirical_size_reselected_project_dd_ci95_high",
        "average_power_reselected_project_dd",
        "mc_reliability",
        "mc_reps",
        "mc_fast_mode",
        "ci_method",
        "mc_boundary"
    ],
    "value": [
        empirical_size_dd,
        empirical_size_dd_ci_low,
        empirical_size_dd_ci_high,
        avg_power_dd,
        mc_reliability,
        MC_REPS,
        MC_FAST_MODE,
        "Wilson",
        (
            "MC_REPS below 100 is debugging only; no formal size/power interpretation"
            if MC_REPS < 100
            else "usable for preliminary finite-sample interpretation"
        )
    ]
})

hl_mc_reliability_report.to_csv(
    OUTPUT_DIR / "39_hong_lee_monte_carlo_reliability_report.csv",
    index=False,
    encoding="utf-8-sig"
)

display(mc_validation_summary)
display(hl_mc_reliability_report)


In [ ]:
# 评估蒙特卡洛模拟收敛性与极限分布数值稳定性
def mc_diagnostic_summary(df, label):
    if "status" not in df.columns:
        return pd.DataFrame({
            "item": ["label", "n_ok", "issue"],
            "value": [label, 0, "missing_status_column"]
        })

    ok = df[df["status"] == "ok"].copy()

    if len(ok) == 0:
        return pd.DataFrame({
            "item": ["label", "n_ok", "issue"],
            "value": [label, 0, "no_successful_replications"]
        })

    out = pd.DataFrame({
        "item": [
            "label",
            "n_ok",
            "convergence_flag_zero_share",
            "mean_std_resid_sq_avg",
            "mean_std_resid_sq_min",
            "mean_std_resid_sq_max",
            "aic_avg"
        ],
        "value": [
            label,
            len(ok),
            np.mean(ok["convergence_flag"] == 0) if "convergence_flag" in ok.columns else np.nan,
            ok["std_resid_mean_sq"].mean() if "std_resid_mean_sq" in ok.columns else np.nan,
            ok["std_resid_mean_sq"].min() if "std_resid_mean_sq" in ok.columns else np.nan,
            ok["std_resid_mean_sq"].max() if "std_resid_mean_sq" in ok.columns else np.nan,
            ok["aic"].mean() if "aic" in ok.columns else np.nan
        ]
    })

    return out

mc_size_diag = mc_diagnostic_summary(mc_size_results, "mc_size")
mc_power_diag = mc_diagnostic_summary(mc_power_results, "mc_power")

mc_diagnostics = pd.concat([mc_size_diag, mc_power_diag], axis=0, ignore_index=True)

mc_diagnostics.to_csv(
    OUTPUT_DIR / "40_monte_carlo_convergence_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_diagnostics


In [ ]:
# 将最优带宽选择与蒙特卡洛模拟实验数据导出至Excel
mc_settings_summary = pd.DataFrame({
    "item": [
        "MC_FAST_MODE",
        "MC_REPS",
        "MC_T",
        "MC_BURN",
        "MC_ALPHA",
        "MC_RESELECT_DD_P",
        "ci_method",
        "mc_interpretation_boundary",
        "sample_selected_p_boundary",
        "project_selector_boundary"
    ],
    "value": [
        MC_FAST_MODE,
        MC_REPS,
        MC_T,
        MC_BURN,
        MC_ALPHA,
        MC_RESELECT_DD_P,
        "Wilson",
        (
            "debugging_only_MC_REPS_below_100"
            if MC_REPS < 100
            else "usable_for_preliminary_finite_sample_interpretation"
        ),
        "sample-selected p-value is post-selection descriptive, not a fixed-p p-value",
        "project-level penalized selector, not Hong-Lee Section 6 exact plug-in selector"
    ]
})

hl_extended_excel_file = OUTPUT_DIR / "hong_lee_project_dd_and_monte_carlo_results.xlsx"

with pd.ExcelWriter(hl_extended_excel_file, engine="openpyxl") as writer:
    mc_settings_summary.to_excel(writer, sheet_name="mc_settings", index=False)

    hl_p_candidates.to_excel(writer, sheet_name="bandwidth_candidates_full", index=False)
    hl_p_candidates_mc.to_excel(writer, sheet_name="bandwidth_candidates_mc", index=False)

    hl_dd_results_df.to_excel(writer, sheet_name="sample_selected_results", index=False)
    hl_dd_ranked.to_excel(writer, sheet_name="sample_selected_ranked", index=False)
    hl_bandwidth_comparison.to_excel(writer, sheet_name="bandwidth_comparison", index=False)

    hl_final_health_report.to_excel(writer, sheet_name="final_model_health", index=False)
    hl_position_usage_dd.to_excel(writer, sheet_name="position_after_dd", index=False)

    mc_size_results.to_excel(writer, sheet_name="mc_size_results", index=False)
    mc_size_summary.to_excel(writer, sheet_name="mc_size_summary", index=False)

    power_dgp_config_df.to_excel(writer, sheet_name="power_dgp_config", index=False)
    mc_power_results.to_excel(writer, sheet_name="mc_power_results", index=False)
    mc_power_summary.to_excel(writer, sheet_name="mc_power_summary", index=False)

    mc_validation_summary.to_excel(writer, sheet_name="mc_validation", index=False)
    hl_mc_reliability_report.to_excel(writer, sheet_name="mc_reliability", index=False)
    mc_diagnostics.to_excel(writer, sheet_name="mc_diagnostics", index=False)

print("Saved:")
print(hl_extended_excel_file.resolve())

display(mc_settings_summary)


In [ ]:
# 波动率高阶特征检验与模型选择项目最终总结白皮书报告
print("========== Hong-Lee 扩展模块 README 段落模板 ==========\n")

print("------------------------------------------------------------")
print()
print("在固定带宽 Hong-Lee M(p) 检验基础上，本文进一步加入 project-level penalized bandwidth selector。")
print("该选择器在多个候选 p 上计算 robust M(p)，并使用惩罚项降低过度选择复杂 bandwidth 的风险。")
print("需要强调的是，该模块是面向白糖项目的惩罚式 selector，")
print("不是 Hong and Lee (2017) Section 6 原文 plug-in bandwidth 的逐字复刻。")
print()
print(f"固定主带宽 p = {HL_P_MAIN:.4f}。")
print(f"真实样本 sample-selected p = {float(hl_dd_selected['p']):.4f}。")
print(f"固定主带宽判断为：{fixed_main_decision}。")
print(f"sample-selected p 判断为：{sample_selected_p_decision}。")
print(f"多带宽稳定性判断为：{bandwidth_stability}。")
print(f"最终 bandwidth conclusion 为：{final_bandwidth_conclusion}。")
print()

print("需要注意：sample-selected p 的 p-value 属于 post-selection descriptive p-value。")
print("因为 p 是基于同一真实样本选择出来的，不能简单等同于固定 p 下的标准 p-value。")
print("因此本文在 Monte Carlo 中额外执行 re-selected project-level data-driven p，")
print("用于评估该选择流程在当前样本长度下的有限样本表现。")
print()

print("Monte Carlo 部分属于离线验证模块，不进入每日交易主流程。")
print("Size simulation 使用 fitted EGARCH(1,1)-t 作为 H0 DGP；")
print("Power simulation 使用 GJR / threshold volatility 与 regime-switching volatility 作为 H1 DGP。")
print("每次 Monte Carlo 复制中，本文同时计算 sample-selected-p Hong-Lee、")
print("re-selected project-level data-driven-p Hong-Lee、Ljung-Box(z)、Ljung-Box(z²) 与 ARCH-LM。")
print()

print("Monte Carlo 拒绝率置信区间采用 Wilson score interval，而不是 Wald interval。")
print("因此，即使出现 0/10 次拒绝，95% CI 上界也不会错误地等于 0。")
print("0/10 只表示样本拒绝率为 0，不表示真实拒绝概率精确为 0。")
print()

if MC_REPS < 100:
    print(f"本轮 MC_REPS = {MC_REPS}，仅用于代码调试，不用于正式统计结论。")
    print("因此，size / power 结果统一标记为 debugging_only，")
    print("不解释为 too_conservative、too_aggressive、low_power、moderate_power 或 high_power。")
else:
    print(f"本轮 MC_REPS = {MC_REPS}，可用于初步有限样本判断。")
    print("但若要论文级 size / power 表格，仍建议使用 MC_REPS = 500–1000。")
print()

print("Power 解释边界：")
print("GJR / threshold volatility DGP 可能被 EGARCH(1,1)-t 部分吸收，")
print("因此 GJR 场景下 power 不高，不能简单解释为 Hong-Lee 无效。")
print("Regime-switching volatility 对 EGARCH 更可能构成结构性错设，")
print("其 power 结果对模型失效识别更有参考价值。")
print()

print("Monte Carlo size summary:")
display(mc_size_summary)
print()

print("Monte Carlo power summary:")
display(mc_power_summary)
print()

print("Monte Carlo reliability:")
display(hl_mc_reliability_report)
print()

print("交易应用上，bandwidth selector 用于减少人为 p 选择风险；")
print("Monte Carlo size/power 用于判断 Hong-Lee model health flag 在当前样本长度下是否可靠。")
print("二者均不提供多空方向信号，只影响 EGARCH 风险模型的可信度、VaR 使用强度和仓位上限。")
print()

print("当前实现属于白糖单变量项目级复现。")
print("若要完整复刻整篇论文，还需继续扩展到论文中的全部 DGP、全部样本长度、全部传统检验对照、")
print("完整 Monte Carlo 表格，以及论文 Section 6 原始 data-driven bandwidth 公式。")
